# OCR a ticker's filings

## 1 · Parameters — the only cell you edit

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════════
#  HOSE_MSN — what is true of THIS ticker, MEASURED from disk 2026-09-07
# ═══════════════════════════════════════════════════════════════════════════════════
#  template `corp`  ·  65 quarter(s) filed  ·  **165 `pdf` cells of 195**  ·  28 quarter(s)
#  open (30 open cells)  ·  0 settled absences  ·  0 span operands
#
#      balance_sheet     59 / 65 `pdf`   open: Q1-2010, Q1-2011, Q3-2012, Q1-2013, Q1-2015, Q2-2018
#      income_statement  44 / 65 `pdf`   open: 21 cells — Q2-2011, Q4-2011, Q1/Q2/Q4-2017,
#                                        Q4-2021, Q1/Q2/Q4-2022, Q1..Q4-2023, Q1..Q4-2024,
#                                        Q2/Q3/Q4-2025, Q1-2026
#      cash_flow         62 / 65 `pdf`   open: Q4-2008, Q1-2010, Q1-2017
#
#  ⚠️ **THIS CLONE WAS A BOOTSTRAP NOTEBOOK ON 2026-09-05 AND IS NOT ONE ANY MORE — every
#     conclusion its old header drew from an EMPTY BAND has been withdrawn.** It read
#     *"0 `pdf` cells of 195 · 195 of 195 bands come back EMPTY · THIS IS A BOOTSTRAP, NOT A
#     GAP RUN, AND `BND-1` IS THE WHOLE SHAPE OF IT · this run parses 65 filings and §9 writes
#     0 rows"*. Ten single-quarter runs on 2026-09-06 (2008-Q4 to 2011-Q4) wrote **21 cells**,
#     so `seed_history` now has something to build a band from and there is no loop to escape.
#     **That changes ONE parameter below and the REASON under two more** — `ONLY_MISSING`
#     False to True, and the notes on `FORCE_EMPTY_BAND` and `OVERWRITE`. §8a's rule applied to
#     this file: a withdrawn conclusion is recorded where the next session meets it.
#
#  ⚠️ **AND THE 60-DOCUMENT RUN HAPPENED: 2026-09-07, 317.8 min = 5.30 h, 0 ENGINE ERRORS,
#     21 `pdf` CELLS → 165.** 155 of 180 statements accepted. That run is what makes this one a
#     different KIND of run: the 30 cells still open are **not 30 unreadable filings**, and
#     every one of them was classified from its own recorded refusal, off the run folders, at
#     no OCR cost:
#
#         14  `reconcile: operating profit does not close`   ← FIXED, see below
#          4  the VAS `Mã số` code column read as a figure   ← FIXED, see below
#          4  de-cumulation with no operand (Q2/Q4-2017, Q2/Q4-2022) — downstream of the 14
#          2  `sane` compares a 12-month figure to a 3-month band (Q4-2011, Q4-2021)
#          2  `reconcile: no total assets` (Q1-2011, Q2-2018)
#          1  no `sane` band at all (Q1-2010 cash flow)
#          3  genuine: Q4-2008 cf (closing cash 5 orders off the balance sheet), Q2-2011 is
#             (`no profit before tax`), Q1-2017 cf (`no closing cash balance`)
#
#  ⚠️ **FIX 1 — `JVW-1`: FOUR SPELLINGS OF THE JOINT-VENTURE LINE, AND MISSING IT REFUSES THE
#     WHOLE STATEMENT.** VAS line 24 is a term of `OP_IDENTITY`'s corp entry, carried as OPTIONAL
#     so a parent-only filing that never prints it is not failed — but "optional" protects only
#     the filing that does NOT print the line. A filing that PRINTS it and a parse that cannot
#     map it fails the identity **by exactly that figure**. MSN's 15 refused income statements
#     spell it `lai_tu_cac_cong_ty_lien_ket` (9), `phan_lai_tu_cac_cong_ty_lien_ket` (5),
#     `loi_nhuan_tu_cac_cong_ty_lien_ket` (1) and `phan_lai_tu_cac_cong_ty_lien_ket_13_c` (1);
#     all four scored 0.56-0.70 against the chart's wording and its one alias, under the 0.80 bar.
#     Two aliases added to `ACCOUNT_WORDING` now carry all four (0.806-0.881).
#     ⚠️ **Q1-2024 IS THE PROOF AND IT NEEDED NO OCR.** The refusal read *"components give
#     1.22796e+13 (or -6.21906e+11 …) against a printed 6.26631e+11"*, and both branches are
#     short of exactly this line: 5,254,838 + 574,011 - 1,899,341 + **1,248,537** - 3,579,977
#     - 971,437 = 626,631, the printed figure to the đồng. The PBT identity (626,631 + 7,032 =
#     633,663) and the after-tax one (633,663 - 294,738 + 139,926 = 478,851) close exactly too.
#
#  ⚠️ **FIX 2 — `MSO` A FIFTH TIME: THE CODE COLUMN WITH NO READABLE HEADING.** All four
#     previous `MSO` fixes widen how the words "Mã số" are READ; a filing that prints no readable
#     heading answers none of them, the column survives, and `_first_value` returns every line's
#     ITEM CODE as its figure. MSN's Q1-2010, Q3-2012, Q1-2013 and Q1-2015 balance sheets were
#     refused as `assets 270,000,000 != liabilities + equity 440,000,000` — **270 IS the VAS code
#     for TỔNG CỘNG TÀI SẢN and 440 the code for TỔNG CỘNG NGUỒN VỐN**, scaled by the statement's
#     own unit. Confirmed off `absent_rows`: `tai_san_ngan_han` [100000000, …], `tien`
#     [111000000, …], `hang_ton_kho` [140000000, …] — column 0 the code, column 1 the figure.
#     `PdfParser._code_column_by_value` asks the COLUMN instead of the page: every entry exactly
#     3 digits and never descending, which is a VAS balance sheet's 100→270 / 300→440 numbering
#     and is not something a period column (4-9 digits in Triệu VND, 10-13 in đồng) can be.
#     ⚠️ **ONLY A BALANCE SHEET CAN EVER NEED IT**, which is why all four are one: the income
#     statement and cash flow number 01..70, and `NOTE_MAX_DIGITS = 2` drops a 2-digit column as
#     a note reference before any heading is consulted.
#     ⚠️ **FIVE NEW LAYERS, POSITIONS 102-106 OF 107 — LAST, AND THE POSITION IS THE WHOLE
#     SAFETY ARGUMENT.** Dropping a detected column changes which figure EVERY line carries, so
#     only a statement every strict read and all four `MSO` widenings already refused reaches
#     them; `is_strict` counts the flag and the last strict layer is position 48.
#     ⚠️ **THREE OF THE FOUR ARE VERIFIED BY REPLAY AND THE FOURTH IS NOT — measured off
#     `absent_rows`, no OCR.** Reading column 1 where the gate was shown column 0 closes the
#     balance-sheet identity to the đồng on three:
#         Q1-2010   7,190,076,000,000 == 7,190,076,000,000  (was 270,000,000 vs 441,000,000)
#         Q3-2012  38,949,558,000,000 == 38,949,558,000,000
#         Q1-2015  49,189,579,000,000 == 49,189,579,000,000 — and its PARTS close too,
#                  27,209,961 + 21,979,618 = 49,189,579 (millions)
#     ⚠️ **Q1-2013 IS RAGGED IN A SECOND WAY AND IS NOT CLAIMED.** Its assets row came back
#     `[270000000, None, 40619911000000]` — the code, NOTHING for the current period, then the
#     comparative — while its resources row lost the code entirely. Dropping the code column
#     leaves it with no current-period total, so it would be refused `no total assets` rather
#     than reconciled, and whether a higher-DPI `+codecol` layer reads the missing figure is
#     UNTESTED (§5 rule 2). **Count fix 2 as 3 verified + 1 unknown, never 4.**
#     ⚠️ **AND THE MEASURED ANSWER WAS 2 OF 4, WHICH IS A LESSON ABOUT THE REPLAY AND NOT
#     ABOUT THE FIX — 2026-09-07.** Q3-2012 ✓ and Q1-2015 ✓; Q1-2013 refused as predicted; and
#     **Q1-2010 refused too, which the replay had counted as verified.** The replay read column 1
#     where the gate had been shown column 0 and asked *would the identity close* — it tested the
#     DATA. It never asked *would the detector fire*, which is a question about the column's
#     SHAPE. Q1-2010's data was sound (7,190,076,000,000 both sides) and its shape defeated the
#     detector. **A replay over recorded rows can verify a figure and can never verify a
#     detector**; the two were conflated, and the honest pre-run number was 2-3, not 3.
#
#  ⚠️ **SO 22 OF THE 30 ARE TARGETED, 8 ARE NOT, AND FOUR OF THE 22 ARE NAMED AS AT RISK.**
#     14 (fix 1) + 4 (fix 2) + the 4 de-cumulation cells the 14 unblock = 22 — of which Q1-2013
#     is the unknown above and Q4-2023/24/25 are the `sane` risk below. **The honest range is
#     18 to 22 of 30, and a Q4 or a Q1-2013 refusal is a PREDICTED outcome, not a failed fix.**
#
#  ⚠️ **BOTH FIXES ARE NOW MEASURED END TO END — A 4-DOCUMENT PROBE, 2026-09-07, 29.8 min.**
#     `QUARTERS = ["2010-Q1", "2012-Q3", "2015-Q1", "2024-Q1"]`, everything else as below, and
#     **165 `pdf` cells → 168** with `source` still only `pdf` or `missing`:
#         Q3-2012  balance_sheet    43 items  **[onnx@200+codecol]**   ← fix 2 ✓
#         Q1-2015  balance_sheet    55 items  **[onnx@200+codecol]**   ← fix 2 ✓
#         Q1-2024  income_statement 15 items  **[onnx@200+equity]**    ← fix 1 ✓
#         Q1-2010  balance_sheet    ABSENT after all 105 layers        ← fix 2 abstained
#     ⚠️ **THE LAYER NAMES ARE THE EVIDENCE, NOT THE CELL COUNT.** Fix 2's two land on the
#     layer written for them, and fix 1's lands on `+equity` — the layer where `equity_wording`
#     turns `ACCOUNT_WORDING` on, which is exactly where the new aliases live. Neither could
#     have been won by an unrelated layer.
#
#  ⚠️ **AND Q1-2010 IS A THIRD SHAPE, NOT A FAILURE OF FIX 2 — MEASURED, NOT ASSUMED.** The
#     `+codecol` layers RAN (101-105 of 105) and the refusal list gained no new entry, so the
#     reason at each was a duplicate of `assets 270,000,000 != liabilities + equity 441,000,000`
#     — i.e. the assets figure never changed and the column was never dropped: the detector
#     ABSTAINED. Its rows say why it would: this filing merges part of the numbering into the
#     LABELS (`tong_cong_tai_san_270_100_200`, `tong_cung_nguon_von_440_300_4004449`), so the
#     code column is ragged and a token like `4004449` in it fails the "every entry exactly 3
#     digits" rule. **That is the fail-safe behaviour working as designed** (§5 rule 2) and the
#     cost of it is one cell. Relaxing the rule to tolerate a few non-code entries is a
#     DIFFERENT decision and needs its own measurement — do not make it to win this one cell.
#     ⚠️ Q1-2010 is also one of the three cells with no `sane` band, so it was never a clean
#     test of fix 2 alone.
#     ⚠️ **THE RISK IS `sane`, AND IT IS THE SAME DEFECT AS THE TWO CELLS FIX 1 DOES NOT
#     TOUCH.** Q4-2023, Q4-2024 and Q4-2025 are CUMULATIVE annual filings, so once fix 1 lets
#     them past `reconcile` they are judged on a 12-month figure against a band whose median is a
#     3-month one. That is exactly what refused Q4-2011 (2.87e+12 vs 1.41e+11, **20.35x**) and
#     Q4-2021 (1.15e+13 vs 5.54e+11, **20.76x**) against a bar of 20x — both inside the window
#     the moment the span is divided out. **Expect 19 to 22, not 22**, and read a Q4 refusal as
#     this and not as fix 1 failing.
#     ⚠️ **AND THE BAND CANNOT GROW ITS WAY OUT DURING THE RUN, MEASURED.** `seed_history`
#     probes the income statement on `C_PBT`; **44 `pdf` income statements are on disk and the
#     band is 3 probes at every quarter from 2017 to 2026** (the balance sheet's grows 2→7→8→
#     14→25 over the same span, the cash flow's reaches 23). So no quarter this run accepts
#     widens the band for the next one — the PBT defect below is what holds it at 3.
#
#  ⚠️ **THE RE-RUN HAPPENED AND THIS HEADER'S NUMBERS ARE NOW ITS INPUT, NOT ITS STATE —
#     2026-09-07, 25 documents, 280.9 min = 4.68 h, 0 ENGINE ERRORS. 168 → 184 `pdf` CELLS OF 195,
#     AND 30 OPEN CELLS BECAME 11.** With the 4-document probe: 29 documents, 310.7 min = 5.18 h.
#     Against what each fix was predicted to reach:
#         `JVW-2`  the JV term              14 targeted, **13 recovered**  (Q1-2026 missed)
#         de-cumulation, downstream          4 targeted, **4 recovered**
#         `MSO-5`  the code column           4 targeted, **2 recovered**  (Q1-2010, Q1-2013)
#         total                             22 targeted, **19 recovered** — the stated range was 18-22
#  ⚠️ **ALL THREE CUMULATIVE Q4s FLAGGED AS `SPN-2` RISKS CLEARED** (Q4-2023, Q4-2024, Q4-2025),
#     because the band GREW inside the run: `MERGE_EACH` writes each complete quarter as it goes and
#     every isolated document re-seeds from the fuller disk, so the income band went **3 → 14**
#     (3 at every document up to Q1-2022, then 4 at Q2-2022 — the first writable quarter — and 8 by
#     Q4-2023). The warning was right about the mechanism and too pessimistic about the outcome.
#  ⚠️ **THE ELEVEN STILL OPEN, EACH WITH ITS CODE**: `BSP-1` 2 (Q1-2011, Q2-2018 bs) · `MSO-5`
#     abstentions 2 (Q1-2010, Q1-2013 bs) · `SPN-2` 2 (Q4-2011, Q4-2021 is) · genuine 3 (Q4-2008 cf,
#     Q1-2017 cf, Q2-2011 is) · Q1-2010 cf, still bandless · Q1-2026 is.
#  ⚠️ **Q1-2026 IS A SECOND DEFECT ON THE JV LINE AND IT IS ORDER-SHAPED, NOT WORDING-SHAPED.**
#     Its statement closes to the đồng — 7,906,027 + 425,056 - 1,894,801 + **1,341,193** - 4,089,684
#     - 1,369,051 = 2,318,740 — and the missing term is `lai_tu_cac_cong_ty_lien_ket`, one of the four
#     spellings `JVW-2` added. **The SCORE is not the blocker: that alias scores 0.808 against it.**
#     The corp chart puts the JV line at order 9 (between `trong_do_chi_phi_lai_vay` and
#     `9_chi_phi_ban_hang`) and MSN's 2026 layout prints it AFTER `chi_phi_quan_ly_doanh_nghiep`,
#     two lines later than the ordered walk expects. **Likely the walk, NOT PROVEN.**
#  ⚠️ **THE CLONE STAYS.** Deletion needs `complete = True` AND `outstanding = 0`; 11 cells are
#     open, and §3 will now resolve `[]` + `ONLY_MISSING = True` to the **10 quarters** carrying them.
#     ⚠️ **A RE-RUN AS IT STANDS WOULD RECOVER NOTHING** — every one of the 11 is named above with
#     a code, and none is waiting on OCR. Fix the code first, or leave them.
#
#  ⚠️ **THE BAND EXISTS NOW — 171 OF 174 OPEN CELLS HAVE ONE, MEASURED PER CELL.**
#     `seed_history(before=<that quarter>)` was asked for every open (quarter, report) pair, the
#     way the run will ask it. Band sizes today, in probes, per entity (`SAN-1` keeps the two
#     apart):
#         balance_sheet     consolidated 3   parent 3
#         income_statement  consolidated 1   parent 1     <- read the next warning
#         cash_flow         consolidated 4   parent 2
#
#  ⚠️ **AND THE ENTITY IS WHY THAT COUNT WAS 1 IN THIS FILE'S FIRST VERSION AND IS 3 NOW —
#     CORRECTED 2026-09-07, MID-RUN, AGAINST AN OBSERVATION.** The first pass counted a cell as
#     banded when EITHER entity had probes. `sane` does not work that way: `SAN-1` keeps the band
#     **per entity**, a consolidated candidate cannot borrow the parent's probes, and
#     `job.plan` gives each filing's entity from the INDEX before anything is parsed. Counted
#     that way, **three cells are bandless, not one**:
#         2008-Q4  cash_flow      parent        0 probes of any entity
#         2010-Q1  balance_sheet  consolidated  0 consolidated, 2 parent — unusable
#         2010-Q1  cash_flow      consolidated  0 consolidated, 1 parent — unusable
#     Every earlier `pdf` row is parent-entity (Q4-2008, Q4-2009, Q2-2010), so 2010-Q1 is the
#     last quarter with nothing of its own to be judged against; from 2010-Q3 on the
#     consolidated probes exist and the question does not arise.
#     ⚠️ **WHAT MADE THIS VISIBLE WAS NOT THE ARITHMETIC BUT AN OBSERVATION.** 2010-Q1's cash
#     flow was ACCEPTED in this run at `onnx@200` with 17 items, and the CSV still read
#     `missing` — which is refusal 2, the empty band, and is also why the 2026-09-06 sweep left
#     it `missing`. A count that cannot explain a row on disk is the wrong count.
#
#  ⚠️ **THE INCOME-STATEMENT BAND IS THREE PROBES AGAINST 44 `pdf` ROWS, AND IT IS A WEAK
#     GUARD WEARING A PASS.** `sane` takes the MEDIAN and accepts median/20..median*20, so a
#     3-probe band is a real guard but by far the widest of the three. It is 3 not because MSN
#     has three income statements but because `seed_history`'s probe column is PBT (`C_PBT`) and
#     **41 of the 44 `pdf` income statements carry no mapped PBT**. Measured per open cell:
#     the band is 3 at EVERY quarter from Q1-2017 to Q1-2026 while the balance sheet's grows
#     2 → 7 → 8 → 14 → 25 and the cash flow's reaches 23 over the same span.
#     ⚠️ **THE CAUSE IS A LABEL VOCABULARY, AND IT IS NOT ONE OF THE TWO FIXES ABOVE.**
#     `C_PBT` names six exact spellings; MSN's parse yields 25+ variants that all contain
#     `truoc_thue` and match none of them. `reconcile` still passes the statement, because
#     `get(self.C_PBT, *self.PBT)` has a TEXT fallback that finds the figure — but `mapped` has
#     no entry for the column, so the accepted row reaches disk with the PBT cell EMPTY.
#     Q3-2022 is the clean case: `loi_nhuan_truoc_thue = 908,892,000,000` passed the gate and is
#     in no column of any CSV. Corpus scope, measured: MSN 31/35 (88.6%), VIC 15.3%, GAS 9.3%,
#     FPT 8.5%, BSR 7.1%, banks 0-6.8% — ~58 statements.
#     ⚠️ **FIXING IT WOULD DISSOLVE THE `sane` SPAN PROBLEM WITHOUT TOUCHING `sane`**, because
#     a 44-probe band contains Q4 figures too and its median stops being a lone small quarter.
#     It is NOT fixed here: the alias that would carry it is gated behind `equity_wording`, and
#     ungating it re-scores which account claims which row on every income statement in the
#     corpus. That needs a corpus measurement first, not a re-run.
#     Do not read "27 of 30 have a band" as "27 of 30 are guarded to the same strength".
#
#  ⚠️ **Q4-2011 IS A FALSE PERMANENT ABSENCE, AND THE GUARD THAT CAUGHT IT IS WHY IT IS STILL
#     OPEN.** Run `20260906-014404` reported all three statements *"no such statement on any
#     page of this filing"* — the ONE reason `settled_absences` treats as permanent — while
#     logging **51 engine errors**: `cuDNN ... HOST_ALLOCATION_FAILED`, then `CUDA failure 2:
#     out of memory` on `onnx@200`, `@300`, `@400` and on. That is `VCR-1` exactly: a layer
#     raises, the layers behind it re-map an EMPTY cached parse, an empty parse has no pages,
#     and a machine failure writes itself into the record as a fact about the FILING.
#     ✅ `settled_absences` skips any document carrying `engine_errors`, so it settled NOTHING
#     for MSN (`settled = {}` today) and all three cells stay open. **The record is
#     contaminated; the plan is not.** The document is `FY-2011_..._da_kiem_toan.pdf`, an
#     audited annual, and it is 5.1 MB.
#
#  ⚠️ **AND THAT IS WHY `ISOLATE_DOCUMENTS`' RETRY MATTERS ON THIS RUN AND DID NOT LAST TIME.**
#     The 2026-09-06 runs were ONE QUARTER EACH, so `run_batch`'s engine-error retry
#     (`RETRIES = 2`, `CAFEF_ONNX_REC_BATCH` 32 then 12 — the one VRAM lever that does not
#     change what is read, measured giving an IDENTICAL `rows_sha`) had a single document to
#     protect and Q4-2011 still ended on 51 raises. This run is 60 documents through
#     `run_batch`, so the retry is live throughout — and `VRAM_FLOOR_MB` below stays at the
#     generic 2600 deliberately: the floor is checked BEFORE a document and cannot prevent a
#     spike DURING one, so raising it would delay the start without buying the guarantee. If
#     Q4-2011 raises again it is the retry that has to answer for it, not the floor.
#
#  ⚠️ **FOUR CELLS WERE REFUSED ON ARITHMETIC, NOT ON CLASSIFICATION — every one still
#     winnable, and each names its own cause** (`SHOW_ABSENT_ROWS` prints the rows behind them):
#         2008-Q4  cash_flow        closing cash 48,733,349,000,000 against the balance sheet's
#                                   cash line 586,133,003 OF THE SAME FILING — five orders out.
#                                   ⚠️ The free cross-check working (§6): `reconcile` caught an
#                                   OCR magnitude error no ratio test could have seen.
#         2010-Q1  balance_sheet    assets 270,000,000 != L+E 441,000,000 at `onnx@200`, and
#                                   != 440,000,000 at `onnx@200+pad6+components` — TWO distinct
#                                   reasons, which is why `settled_absences` refuses to settle it
#         2011-Q1  balance_sheet    `no total assets`
#         2011-Q2  income_statement `no profit before tax`
#
#  ⚠️ **THREE `pdf` ROWS ON DISK ARE THIN AND SHOULD BE SCREENED BEFORE THEY ARE QUOTED OR
#     LEANED ON AS A BAND** — ACCEPTED rows, not refused ones, which is the class `P47`/`P48`
#     exist for and precisely the class `ONLY_MISSING` cannot see:
#         income_statement Q4-2010    **2 mapped items** — excluded from the band by
#                                     MIN_ITEMS_FOR_HISTORY, and `pdf` all the same
#         cash_flow        Q1-2011    7 mapped items (also excluded)
#         balance_sheet    Q4-2008   13 mapped items against Q4-2009's 51
#
#  ⚠️ **FIVE QUARTERS HAVE NO FILING AT ALL AND `missing` IS THE CORRECT, PERMANENT ANSWER**
#     (§5 rule 24): **2009-Q1, 2009-Q2, 2009-Q3, 2014-Q1, 2015-Q3** — 15 cells no run can ever
#     fill. Re-verified against the index today: the 65 filed quarters span 2008-Q4 to 2026-Q1,
#     which is 70 calendar quarters. So the ticker's ceiling is **195 cells of which 180 are
#     reachable**, and 2014-Q1 / 2015-Q3 are ordinary mid-life gaps rather than 2009's
#     "the company only filed annuals back then" shape.
#
#  ⚠️ **FOUR CUMULATIVE QUARTERS CAN NEVER BE DE-CUMULATED, AND THAT IS NOT A DEFECT** — of 31
#     cumulative filings these four need a prior that was NEVER FILED, so the subtraction has no
#     operands and never will:
#         2008-Q4  needs Q1..Q3 2008 (MSN's first filing IS the FY-2008 annual)
#         2014-Q2  needs Q1-2014        2014-Q4  needs Q1-2014
#         2015-Q4  needs Q3-2015
#     The merge KEEPS such a row, labelled `months = 12` (or 6), rather than dropping it: the
#     choice is not *"cumulative now or a quarter later"* but *"cumulative now or nothing,
#     ever"*. ⚠️ Any TTM or ratio built from those four would count a YEAR as a QUARTER, and the
#     `months` column is the only thing that says so. The other 27 stay unwritable until their
#     own priors are on disk — which is why §9 merges one period at a time, OLDEST FIRST, and
#     why a dry run UNDERSTATES what a two-pass merge will write.
#
#  ⚠️ **THE ALTERNATE COUNT IN THE OLD HEADER WAS WRONG BY ~3x AND IS CORRECTED HERE.** It
#     claimed *"51 of the 65 quarters have an alternate — 89 alternates in all, more than any
#     ticker parsed so far"*. Measured through `builder.alternates` — the call `_alternate_retry`
#     (`ALT-1`) actually makes — it is **30 quarters and 30 alternates, 28 of them among the 60
#     outstanding.** The 89 was an ENTITY-BLIND count off the raw index (154 rows over 80
#     periods, 74 of them extra filings); `alternates` **fixes** the entity rather than
#     preferring it, so a fallback can never quietly change which company a row describes.
#     ⚠️ Read §8's origin line before believing a recovered statement's provenance — a row can
#     legitimately name a different filing from the one the document block names, which is what
#     TCB's Q2-2019 needed. ⚠️ And an alternate trades assurance for coverage with NO guard
#     against a RESTATEMENT: BID's unaudited Q4-2016 disagrees with its audited annual by
#     ~2.9 tn and both reconcile.
#
#  ⚠️ **ONE PARENT-COMPANY FILING IS LEFT AMONG THE OUTSTANDING** (`consolidated = False`),
#     reached only because `ALLOW_PARENT = True`. Three of the nine parsed were parent (2008-Q4
#     audited, 2009-Q4 unaudited, 2010-Q2 reviewed) — ⚠️ and note Q1-2010's income statement is
#     CONSOLIDATED while Q2-2010's is PARENT, so *"everything from 2010-Q3 on is consolidated"*
#     (the old header) is not a rule to lean on. Two entities in one column is the defect the
#     `consolidated` column exists to prevent, and `sane` bands per entity for the same reason
#     (`SAN-1`).
#
#  ⚠️ **THE RUNTIME IS MEASURED ON THIS TICKER, ON THIS MACHINE, OVER 60 DOCUMENTS — AND THE
#     ESTIMATE THIS BLOCK CARRIED WAS WRONG.** It read *"8.2 h by document count, 10.2 h by
#     payload, and **4-6 h is the likelier figure**"*. The run came in at **317.8 min = 5.30 h,
#     mean 5.30 min/document, 0 engine errors** — inside that 4-6 h, so the correction is not
#     the number but the CONFIDENCE: the two upper bounds were 1.5x and 1.9x high, and the
#     "likelier figure" was a guess that happened to land. What is measured now is one figure
#     from one full run of this ticker.
#         plan bound          28 documents x 5.30 min  ~= 2.5 h
#         year-granularity    49 documents x 5.30 min  ~= 4.3 h
#     ⚠️ **THE SPREAD IS `_skippable_years`, NOT UNCERTAINTY.** `plan_batch` resolves 28
#     quarters — exactly the 28 carrying a gap — but the run skips a whole YEAR only when every
#     attempted quarter of it is already `pdf` in all three statements, because `_decumulate`
#     takes its priors from THIS RUN. 2009, 2014, 2016, 2019 and 2020 are whole and are dropped;
#     the 14 years that are not hold 49 filed quarters between them. §3 prints which it is
#     before anything is spent.
#     ⚠️ Do not quote a LOCAL-vs-T4 speedup off any of it: four runs of one identical easy
#     document on this machine spanned 50.3-113.3 s, a 2.25x swing with the T4's 69.0 s inside it.
#     ⚠️ **Nothing caps a LOCAL run**, so unlike Kaggle's 12 h it need not be split, and
#     `MERGE_EACH` below means stopping it early keeps every quarter already finished.
#
#  ⚠️ **`CRP-1`: NOTHING FROM THIS TICKER MAY BE QUOTED AS A FUNDAMENTAL.** MSN files on the
#     `corp` chart, where `C_LIABILITIES` does not map — so `reconcile` tests
#     `assets == resources`, true by construction on any page that reads both totals, and never
#     `A = L + E`. `SEC-1`'s section sums, `GTL-1`'s tolerance and `CBS-1`'s balance-sheet
#     cross-check are real gates where there was none; that is not the same thing as a checked
#     balance sheet. ⚠️ 2010-Q1's refusal above shows the `corp` gate is not vacuous — but what
#     it caught was 270,000,000 against 441,000,000, not an `A = L + E` failure.
#
#  ⚠️ **`TPX-1`: `templates.csv` HOLDS ACB, BID AND VCB AND NOT MSN**, so `corp` is resolved by a
#     NETWORK call to CafeF's own fingerprint — measured today, route
#     `detect_template (CafeF fingerprint, over the network)`. On LOCAL that call happens in §3
#     on this machine, every run; the offline route is quietly gone. ⚠️ The 2026-09-06 runs
#     recorded `template_how = override` because they PINNED `TEMPLATE = "corp"`; this notebook
#     leaves it `None` so the route is recorded rather than assumed.
#     ⚠️ **AND MSN IS IN NEITHER FINANCIALS REGISTER** — not `CAFEF_FINANCIALS_TICKERS` (VCB,
#     ACB, BID, VIC and nothing else), not `orchestration/config.json`'s `raw/cafef_financials`
#     partitions. Its `"MSN": true` in that file is under `unified`, which is the PRICE schema
#     and unrelated. So its Dagster asset cannot be materialised and its statements feed no
#     silver ingest: **parsing it changes no table** (§6-2-untricies). This notebook needs
#     neither registration; the Dagster path does.
#
#  ⚠️ **THE SCREENS ARE NOT OPTIONAL HERE — `P47`(b), shipped 2026-09-04.** `sane`'s band is one
#     probe on the income statement and absent on 2008-Q4's cash flow, so
#     `web_scraper.statement_screens` is what stands in for it: run `screen_run` over the run
#     folders BEFORE §9 and hold every flagged (quarter, statement) pair back through
#     `MERGE_REPORTS` and an explicit quarter list — never by editing the artefact. Measured
#     precedent: TCB's 95.5 % "parsed" concealed a 5.3 % wrong-figure rate, and on three of four
#     bootstrap tickers these screens stood between wrong figures and a CSV.
#     ⚠️ The `unit` screen is deliberately NOT among them: it convicted 8 TCB statements and
#     then flagged 32 CORRECT CTG ones. What convicts is the MAGNITUDE, never the unit.
# ═══════════════════════════════════════════════════════════════════════════════════

# ── PARAMETERS — the only cell you edit ───────────────────────────────
ENVIRONMENT = "LOCAL"        # "LOCAL" = parse here | "KAGGLE" = ship it to a T4
                             # ⚠️ LOCAL as asked. Nothing is uploaded, `REHEARSE` is
                             #    ignored, and the 12-hour Kaggle cap does not apply —
                             #    but the card is 4 GiB, so `ISOLATE_DOCUMENTS` below
                             #    is what makes 60 documents possible at all (`GPU-1`),
                             #    and Q4-2011's 51 CUDA-OOM raises on 2026-09-06 are
                             #    what that flag's retry exists to answer (see header).
EXCHANGE    = "HOSE"         # HOSE | HNX | UPCOM
SYMBOL      = "MSN"          # ticker, as CafeF files it

# WHICH QUARTERS — A LIST, AND NOTHING ELSE. Each entry is YYYY-QQ; "2026-Q4" and the
# zero-padded "2026-04" are the same quarter, folded once at the edge.
#   []                     ->  EVERY quarter this ticker files  (⚠️ ~70 documents, hours).
#                              ⚠️ Safe on this 4 GiB card ONLY because `ISOLATE_DOCUMENTS`
#                              is on; the same list in one process died at document 4
#                              (`GPU-1`). `ONLY_MISSING` below narrows it to the gap.
#   ["2014-Q4", "2015-03"] ->  exactly these quarters, and nothing else.
# ⚠️ The repo-native "Q3-2014" is REFUSED rather than quietly accepted: a typo has to report
#    itself as a typo, not as a quarter CafeF does not file. A STRING is refused for the same
#    reason — a bare "2014-Q4" with the brackets forgotten included; §2 has the measurement.
QUARTERS = []   # ⚠️ THE DEFAULT CHANGED 2026-09-03 AND IT IS THE EXPENSIVE DIRECTION. This
                # read "OUTSTANDING", which resolved to the GAP and raised when there was none;
                # `[]` is every quarter the ticker files — ~70 documents and hours — on a
                # notebook somebody just pressed run on. §2 and §3 both print which it is and
                # how many documents, before anything is spent.
                #   the old default back:  ONLY_MISSING = True
                #   one quarter:           QUARTERS = ["2014-Q4"]
                ############################################################################
                # ⚠️ **EMPTY HERE, AND `ONLY_MISSING = True` BELOW IS WHAT NARROWS IT.**
                #    ⚠️ **§3 RESOLVES THIS TO 28 QUARTERS NOW, NOT 60** — measured with
                #    `plan_batch` today, and they are exactly the 28 carrying one of the
                #    30 open cells. 37 of the 65 filed quarters are complete in all three
                #    statements and are dropped before any OCR.
                #    ⚠️ **THE RUN MAY STILL PARSE UP TO 49**, because
                #    `_skippable_years` skips a YEAR and not a quarter — see the runtime
                #    block in the header.
                #    ⚠️ To exercise ONE fix rather than both, name its quarters instead;
                #    `ONLY_MISSING` would open all 28:
                #      fix 2, the three REPLAY-VERIFIED: ["2010-Q1", "2012-Q3", "2015-Q1"]
                #      fix 2, the unknown:               ["2013-Q1"]
                #      fix 1, the proof case:            ["2024-Q1"]
                #    ⚠️ **A NAMED-QUARTERS RUN COSTS ~5 MIN A DOCUMENT AGAINST 2.5-4.3 h**, so
                #    it is how to settle either fix before spending the whole gap on it.
                ############################################################################

# ⚠️ NARROW AN EMPTY `QUARTERS` TO WHAT IS STILL MISSING — read ONLY when the list is empty,
#    and it is what the retired "OUTSTANDING" sentinel became.
#   False -> every quarter the ticker files, which is what `[]` says above.
#   True  -> exactly the quarters §3 finds still `missing` AND still winnable, plus the span
#            operands they need. Resolved from the three statement CSVs and the PDF index,
#            printed before anything is spent, and it RAISES rather than falling through to
#            "every quarter" when there is nothing left to do.
ONLY_MISSING = True   # ⚠️ **TRUE, AND THIS IS THE ONE PARAMETER 2026-09-06's RUNS CHANGED.**
                      #    §1a's rule: `[]` + True when the ticker ALREADY HAS statement CSVs
                      #    — parse the GAP, not the ticker — and `[]` + False only when it has
                      #    none. On 2026-09-05 MSN had none, so this clone shipped False; it
                      #    has **165 `pdf` cells** now, so True is the correct half of that rule.
                      #    ⚠️ IT NARROWS 65 DOCUMENTS TO 28, AND ON THIS RUN THAT IS MOST OF
                      #    THE COST — it was 65 to 60 last time, which was almost nothing.
                      #    What it buys is that the resolved list is a MEASUREMENT of the gap
                      #    rather than a coincidence of two defaults, and that it **RAISES**
                      #    when nothing is left instead of silently opening all 65.
                      #    ⚠️ **AND IT CANNOT SEE A WRONG `pdf` ROW** — it resolves cells that
                      #    read `missing`; the three THIN accepted rows in the header look
                      #    finished to it. That is `EQW-1`/`NST-1`/`PAR-1`'s class, and the
                      #    screens that find it are `P47`/`P48`, not this flag.

# ⚠️ **`open` IS NOT `WINNABLE`, AND §3 CANNOT TELL YOU WHICH — so read this before setting
#    ONLY_MISSING = True on a ticker you have already run.** `settled_absences` records one
#    reason and one only: `no such statement on any page of this filing`, which is a verdict
#    on the DOCUMENT and therefore permanent. Every OTHER refusal — a total that will not
#    balance, an identity that does not close, a magnitude the guard rejected — is reported
#    as `open — a re-run could still win it`, because a later layer or a fixed anchor could
#    in principle overturn it. Re-running one costs the FULL cascade to return the same word.
#
# ⚠️ **AND WHEN ONE COMES BACK `absent` TWICE, THE FIRST THING TO CHECK IS THE PDF INDEX,
#    NOT THE LAYERS.** `documents()` returns ONE filing per period and a quarter can have
#    several. Measured 2026-09-04 on TCB's Q2-2019: its closing cash balance is printed under
#    the company's round stamp in the AUDITED consolidated filing, so the recogniser returns
#    a different wrong figure at 200, 300, 400+pad6, 500 and 600 dpi and never the printed
#    one — and the REVIEWED consolidated filing of the same quarter is a different scan that
#    reads the whole tail cleanly at layer 1. *No OCR configuration can read this figure* was
#    measured, true, and written up as *this quarter cannot be parsed*, which is a claim about
#    a different thing. `_alternate_retry` (`ALT-1`) now tries the others automatically; §8
#    prints which filing each recovered statement came from.
#    So: read §8's `absent_reasons`, check the index for a second filing, and write whatever
#    you settle down where a reader meets it BEFORE spending the cascade again — this comment
#    or the per-ticker notebook. §6-2-septquadragies is the same lesson for the settled kind:
#    *a measurement that exists only as prose is one the next session cannot act on.*
#
# ⚠️ **AND A `SETTLED` CELL IS NOT PROOF EITHER — `SET-2`, measured 2026-09-04.** The one
#    reason `settled_absences` treats as PERMANENT, `no such statement on any page of this
#    filing`, is a verdict on the PAGE CLASSIFIER and reads as one on the document. TCB's
#    Q1-2017 and Q3-2017 print the notes title AND the notes form code on the cash flow's
#    FIRST page, so no cash-flow page is found and both were recorded as filings containing
#    no cash flow — page 8 of Q1-2017 prints "LƯU CHUYỂN TIỀN THUẦN TỪ HOẠT ĐỘNG KINH
#    DOANH" over 67 figures. §3 DROPS such a cell before any OCR, so re-trying one means
#    naming its quarter in QUARTERS explicitly.

# ⚠️ What to do about a quarter ALREADY on disk:
#   False -> FILL THE GAPS. One reading `pdf` in all three statements is dropped before any OCR
#            (and before it is uploaded); a figure that DIFFERS is never written over it.
#   True  -> re-parse every selected quarter and let the result replace what disk holds.
#   ⚠️ To replace ONE wrong row use REPAIR below, never this — see the note there.
# ⚠️ TRUE IS REQUIRED BY `SPAN_OPERANDS`, and that is the only reason it is the default here:
#    a span operand is BY DEFINITION a quarter already reading `pdf`, so with False it is
#    dropped before any OCR and the Q4 it unblocks stays unwritable. It is safe in this
#    combination ONLY because MERGE_INTO_CSV is off — `force_differs` follows OVERWRITE into
#    the automatic per-quarter merge, and never into §9's.
OVERWRITE = False   # ⚠️ FALSE HERE, AGAINST THE GENERIC DEFAULT, AND IT IS MEASURED
                    #    — BUT THE REASON IS NO LONGER THE ONE THIS CLONE SHIPPED WITH.
                    #    It read *"a span operand is by definition a quarter already
                    #    reading `pdf` and this ticker has none — 0 `pdf` cells of 195"*.
                    #    There are **165 `pdf` cells** now, and 37 quarters complete in
                    #    all three statements, so that argument is gone.
                    #    ⚠️ **THE ANSWER IS UNCHANGED AND NOW IT IS A REAL CHOICE.**
                    #    `plan_batch(span_operands=True)` still reports **0 operands** —
                    #    measured today, and the reason is that all 5 complete quarters
                    #    already record `months = 3` on their income statement, so no
                    #    outstanding Q4 is blocked by a blank span in another row. With
                    #    no operand to protect, False is strictly safer: a quarter
                    #    complete in all three is dropped before any OCR, and a figure
                    #    that DIFFERS from a good `pdf` row is never written over it.
                    #    ⚠️ **AND FALSE IS NOW LOAD-BEARING, WHERE IT USED TO BE FREE.**
                    #    There ARE 165 cells on disk to differ from — among them the three
                    #    THIN ones in the header — and this run's `sane` band is
                    #    `seed_history`'s reconstruction, not the band the 2026-09-06 runs
                    #    accumulated, so the two escalate DIFFERENTLY and a seeded run can
                    #    win on an earlier, poorer layer (ACB, 2026-08-30: 33 items at
                    #    `onnx@200+relax` on disk against 19 at `onnx@200` seeded).
                    #    Repair a named row with `REPAIR` below, never with this knob.
                    #    §2 refuses SPAN_OPERANDS without it.

# UPSERT the accepted statements into raw_data/.../statements/*.csv, through `pdf_ocr_merge`:
# it BACKS THE THREE CSVs UP FIRST, prints every changed cell, and refuses four things it
# cannot judge — a statement whose `sane` band was empty, a figure that DIFFERS from a good
# `pdf` row, a cumulative income statement whose priors it cannot subtract, and ⚠️ a document
# any of whose layers RAISED (`VCR-1`: an exception measures the MACHINE, not the filing, so
# whatever won the cascade won by default).
# ⚠️ OFF, AND §9 DOES THE UPSERT — for two independent reasons:
#    (1) the automatic path passes `force_differs = OVERWRITE`, which is True above;
#    (2) `merge_run` PLANS THE WHOLE FOLDER AGAINST DISK AND WRITES AFTERWARDS, so one call
#        would decide a Q4 while the Q3 span it depends on is still whatever disk held when
#        the call started. §9 merges one period at a time, oldest first, which is the only
#        shape in which a span operand reaches the quarter it exists to unblock.
# ⚠️ OFF NO LONGER MEANS "THE CSVs ARE LEFT ALONE" — §9 WRITES BY DEFAULT since
#    2026-09-04 (`MERGE_APPLY = True` below). What this flag decides now is only WHICH
#    path does the upsert: the pull's one blanket call with `force_differs = OVERWRITE`,
#    or §9's ordered unforced one. Leave it off — §9 is the safer of the two, not the
#    slower one.
MERGE_INTO_CSV = False

# ⚠️ WRITE A STATEMENT WHOSE `sane` BAND WAS EMPTY — it lifts a real guard (`BND-1`).
#   True  -> write it anyway.
#   False -> keep the guard.
# ⚠️ **IT NO LONGER DECIDES WHETHER A NEW TICKER CAN START, AND THAT CHANGED 2026-09-06.** A
#    quarter whose filing produced ALL THREE statements is written band or no band, by both
#    writers, because refusing it was `BND-1`'s loop rather than a guard — see MERGE_EACH.
#    What this flag still governs is everything that gate does NOT cover: a filing that
#    produced two statements of three, which is a judgement about THAT filing and stays the
#    operator's. False is right for a ticker with history on disk, and now also for a
#    bootstrap — the bootstrap no longer needs it.
FORCE_EMPTY_BAND = False   # ⚠️ THE JUDGEMENT CALL ON THIS TICKER, AND THE MEASUREMENT
                           #    BEHIND IT IS THE ONE THAT FLIPPED SINCE 2026-09-05. This
                           #    clone shipped saying **195 of 195** open cells come back
                           #    with an EMPTY band, so the run would parse 65 filings and
                           #    §9 would write **0 rows** — `BND-1`'s closed loop.
                           #    ⚠️ **TODAY IT IS 3 OF 30** — asked per open (quarter,
                           #    report) pair the way the run asks it, and counted against
                           #    each filing's OWN entity, because that is what `sane`
                           #    compares (`SAN-1`). **27 of 30 have a band.** ⚠️ **THE THREE
                           #    ARE THE SAME THREE AS BEFORE THE 60-DOCUMENT RUN** — 2008-Q4's
                           #    cash flow (0 probes of either entity) and BOTH of 2010-Q1's
                           #    open cells (0 consolidated against 2 and 1 parent, which
                           #    `SAN-1` forbids borrowing). 144 new cells moved none of them,
                           #    because all three are the OLDEST open cells in the chain and
                           #    a band is built only from what precedes. See the header for
                           #    why the entity is the whole reason.
                           #    ⚠️ **SO FALSE COSTS ALMOST NOTHING NOW, AND IT IS NOT A
                           #    BOOTSTRAP DECISION AT ALL.** Since 2026-09-06 a quarter
                           #    whose filing produced ALL THREE statements is written band
                           #    or no band, by both writers; what this flag still governs is
                           #    a filing that produced TWO of three.
                           #    ⚠️ **SO EXACTLY ONE CELL IS AT STAKE, AND IT IS NAMED:
                           #    2010-Q1's CASH FLOW.** Of the three bandless cells, two are
                           #    absent on ARITHMETIC and no band would help them — 2008-Q4's
                           #    cash flow (closing cash five orders off the balance sheet's
                           #    own cash line) and 2010-Q1's balance sheet (assets
                           #    270,000,000 vs L+E 441,000,000, twice). The third PARSES:
                           #    2010-Q1's cash flow came back at `onnx@200` with 17 items in
                           #    both the 2026-09-06 and the 2026-09-07 runs, and is refused
                           #    only for having no consolidated band.
                           #    ⚠️ **DO NOT FLIP THIS TO GET IT.** Its own filing produced two
                           #    of three, so the whole quarter is held either way, and the
                           #    flag would lift the guard for every other statement in a
                           #    60-document run. The scoped way is a later one-quarter run
                           #    over 2010-Q1 alone, AFTER the free cross-check: that cash
                           #    flow's OPENING balance must equal Q4-2009's CLOSING, which is
                           #    already `pdf` on disk — no OCR, no network (§6).
                           #    ⚠️ Flipping it lifts the one guard that reads a MAGNITUDE,
                           #    and `web_scraper.statement_screens` then has to replace it BY
                           #    HAND before §9 (`P47`(b); the header has the precedent).

# ⚠️ THE ONNX-ONLY CASCADE — 53 layers of 55, and it is about REPRODUCING, not about speed.
#   True  -> drop `tesseract@200` and `tesseract@400+relax`.
#   False -> the full cascade as shipped.
# ⚠️ `tesseract@200` IS LAYER 4 OF 55 HERE AND DOES NOT EXIST ON A KAGGLE WORKER (`TSS-1`,
#    CLAUDE.md §6-2-quinquagies). So it can win a statement twenty onnx layers would have read
#    better, and every ticker bootstrapped on a T4 carries rows produced by the 53-layer
#    cascade — a local re-parse under the full 55 is a DIFFERENT PROCEDURE and reports the
#    difference as DIFFERS. Measured on BSR Q3-2019: `tesseract@200` read
#    361,884,738 where the Kaggle `onnx@300+tail` row reads 361,884,738,267.
ONNX_ONLY = True   # ⚠️ TRUE, AND ON THIS TICKER THAT IS MEASURED RATHER THAN INHERITED.
                   #    §1a's rule is "True for any ticker whose rows were bootstrapped on a
                   #    T4"; MSN's 21 rows were bootstrapped LOCALLY, and the reason is the
                   #    same one either way — every one of the ten 2026-09-06 runs recorded
                   #    **100 requested layers, 0 of them tesseract**
                   #    (`layers_are_the_full_cascade = False`). So the rows this run is
                   #    compared against were produced by the onnx-only cascade, and running
                   #    the full one now would report the difference as DIFFERS (`TSS-1`).
                   #    ⚠️ The guide says "53 of 55"; the cascade this repo builds today is
                   #    **105 onnx layers of 107** — it was 100 of 102 for the 2026-09-07 run
                   #    and grew by the five `+codecol` layers named in the header. Trust the
                   #    run folder's `layers`, not the prose — `metadata.json` records exactly
                   #    what was asked for.
                   #    ⚠️ **AND THIS RUN'S COUNT DIFFERING FROM THE ROWS IT COMPARES
                   #    AGAINST IS EXPECTED AND IS NOT `TSS-1`.** The five new layers sit at
                   #    positions 102-106, past every layer any `pdf` row on disk was won at,
                   #    so a cell that already parses cannot reach them.

# ⚠️ PULL IN THE QUARTERS A CUMULATIVE Q4 NEEDS AS OPERANDS (`QUARTERS = []` with
#    ONLY_MISSING = True only — the other two modes already name every quarter they are going
#    to open, so there is nothing left for this to add).
#   A Q4 income statement is the YEAR, and the standalone quarter is FY − (Q1+Q2+Q3). The
#   merge will only subtract a prior whose span is a KNOWN three months, and most of the
#   corpus predates the `months` column — so the priors read `unrecorded`, a blank is NOT 3
#   (§5 rule 2), and the Q4 is refused however well it parsed.
# ⚠️ MEASURED: CTG carried SEVEN such Q4 income statements on 2026-09-02, every one of them
#    parsed and none of them writable, blocked by a blank column in ANOTHER ROW. Re-parsing a
#    prior moves no figure — an unchanged reading goes through the merge's `fills_span`
#    branch, which writes the span and nothing else.
SPAN_OPERANDS = False   # ⚠️ FALSE: `plan_batch(span_operands=True)` returns **0
                        #    operands** for MSN today (see OVERWRITE above for why — all
                        #    5 complete quarters already record `months = 3`), and §2
                        #    raises on SPAN_OPERANDS without OVERWRITE.
                        #    ⚠️ **AND THIS IS NOW A RUN IT WOULD ACTUALLY READ**, which it
                        #    was not when this clone shipped: it reads only when QUARTERS
                        #    is empty AND `ONLY_MISSING` is True, and both are true above.
                        #    So False here is a measurement, not a mode that skips it.

# ⚠️ ONE PROCESS PER DOCUMENT — what makes a WHOLE-TICKER run possible on a 4 GiB card.
#   True  -> `pdf_ocr_batch.run_batch`: a fresh process per filing, and it waits for the card
#            to have VRAM_FLOOR_MB free before each one.
#   False -> `pdf_ocr_job.run` parses every filing in THIS process. Right for one quarter.
# ⚠️ MEASURED 2026-09-02: 18 documents in one process cleared three filings and then every
#    `onnx@*` layer raised `CUDA failure 2: out of memory` — 294 of them — and the cascade
#    went on and reported `pdf` for statements it had been unable to read. The same 25
#    documents, one process each, ran with **0 engine errors**. It changes no semantics:
#    `seed_history` re-seeds `sane` from DISK per document and the page cache is per filing.
# ⚠️ The cost is model load, ~10-20 s per document.
ISOLATE_DOCUMENTS = True
VRAM_FLOOR_MB = 2600     # free VRAM one document wants before it starts; a filing peaked at 2.9-3.2 GiB
SHOW_ABSENT_ROWS = True  # §8 prints the rows behind a REFUSED statement — the cause, not the symptom

TEMPLATE     = None      # None = RESOLVE it (templates.csv, then CafeF's fingerprint). ⚠️ Never defaulted to "bank".
ALLOW_PARENT = True      # fall back to the STANDALONE filing where no consolidated one exists
PERIODS      = None      # the repo-native form, e.g. ["Q3-2014"]. Optional, and INTERSECTS with QUARTERS.
LAYERS       = None      # None = the cascade ONNX_ONLY selects, in cascade order
COMPARE      = True      # score every parsed cell against the statement CSV already on disk
NOTES        = ("HOSE_MSN — the 28-quarter / 30-cell gap after the 2026-09-07 run "
                "(165 pdf cells of 195). Exercises two fixes: JVW-1's four spellings "
                "of the VAS line-24 joint-venture term (14 cells) and MSO's code "
                "column recognised by value (4 cells), + 4 de-cumulation cells "
                "downstream. Expect 19-22 of 30: Q4-2023/24/25 are cumulative and "
                "may still hit sane's 20x band, which is what refused Q4-2011 and "
                "Q4-2021")

# ── THE MERGE — as the run goes (§6), then the sweep (§9) ────────────────────────────
# ⚠️ Merging period by period is not a style choice: `merge_run` plans against disk and writes
#    afterwards, so a span recorded for one quarter reaches the NEXT quarter's planner only in
#    the following call. That is the dependency a span operand needs.

# ⚠️ WRITE EACH QUARTER THE MOMENT ITS FILING HAS PRODUCED ALL THREE STATEMENTS, instead of
#    waiting for §9. ⚠️ **LOCAL + ISOLATE_DOCUMENTS ONLY** — on KAGGLE the worker's data root
#    is a payload that dies with the kernel (`pdf_ocr_job.run` refuses to merge there at all),
#    so the write is the pull's; on the one-process path it is `MERGE_INTO_CSV` above.
#   True  -> `run_batch` upserts a finished quarter BETWEEN DOCUMENTS, through the same
#            `merge_run` §9 calls, and `force_differs` is NEVER passed.
#   False -> nothing reaches the CSVs until §9.
# ⚠️ **THE CSV IS CREATED IF THE TICKER HAS NONE, AND NO KNOB IS NEEDED FOR IT** (2026-09-06,
#    by request). `FinancialsBuilder._write` makes the directory and the file; what used to
#    stop a brand-new ticker was not the missing file but refusal 2 — an EMPTY magnitude band,
#    meaning `sane` failed open and the figure passed no guard — and refusing on that is a
#    LOOP, not a guard: no `pdf` row -> no band -> every statement refused -> still no `pdf`
#    row (`BND-1`). So a quarter that clears the gate is written whether or not `sane` had a
#    band, on this path and in §9 alike, and each such row is PRINTED and RECORDED as
#    unguarded — in the run folder's `merge` block, and again in §10.
# ⚠️ **THOSE ROWS PASSED NO MAGNITUDE GUARD. SCREEN THEM BY ARITHMETIC BEFORE QUOTING ANY OF
#    THEM** — two statements agreeing on one figure, a printed subtotal closing. `sane` is what
#    catches an OCR misread by three orders of magnitude (BSR Q3-2019 was read as
#    361,884,738 where another layer reads 361,884,738,267), and on a bootstrap it is not there.
# ⚠️ THE OTHER THREE REFUSALS ARE UNTOUCHED: a figure that DIFFERS from a `pdf` row on disk, a
#    cumulative income statement whose priors cannot be subtracted, and a document any of whose
#    layers RAISED are all still refused.
# ⚠️ **THE REASON IT IS ON: A RUN THAT STOPS HALFWAY KEEPS WHAT IT HAS ALREADY READ.** ⚠️ The
#    measurement is HOSE_FPT, 2026-09-04, and its CAUSE was a knob and not an interrupt — a
#    185-minute round trip over 71 filings accepted 128 of 213 statements, §9 planned 96 WRITEs
#    and 0 of them reached disk because MERGE_APPLY was off. What it measures for THIS flag is
#    the shape the two share: the parse is durable in the run folder and the CSVs are not
#    touched until a later step that may never run (`BND-1` — the work is on disk, the CSV is
#    not, and a green run says nothing about which). `_write` renders to a `.tmp` and
#    `os.replace`s it, so an interrupt can lose the quarter in flight and never one on disk.
# ⚠️ **THE GATE IS THE FILING, NOT THE STATEMENT.** A document that accepted two of three is
#    HELD — and named in the log as it happens — for §9, where you are reading the refusals.
#    The three CSVs of a quarter move together or they do not move.
# ⚠️ **AND IT IS ALSO `SPN-1`'s ORDER, FOR FREE**: the quarters are parsed oldest first, so a
#    span operand is written before the Q4 it exists to unblock is even planned. §9 has to
#    reproduce that order deliberately; here it falls out of the loop.
# ⚠️ IT DOES NOT MAKE §9 REDUNDANT — what it wrote comes back `identical to the row already on
#    disk`, which is a check, and what it held is what §9 picks up.
MERGE_EACH = True

MERGE_TWO_PASS = True
MERGE_REPORTS = None     # ⚠️ WHICH STATEMENTS §9 MAY WRITE. None = all three.
                         # ⚠️ SCOPE IT WHEN YOU ARE REPAIRING ONE CELL. A quarter whose
                         # other two statements are already `pdf` FROM THE SAME filing
                         # has nothing to gain from leaving them writable, and something
                         # to lose: a DIFFERS decided by recency rather than by the
                         # filing. CTG Q4-2014 was written that way on 2026-09-03.
MERGE_APPLY   = True     # ⚠️ THE DEFAULT SINCE 2026-09-04, AND IT IS WHAT MAKES THIS
                         # NOTEBOOK WRITE. It was False, so a run that parsed perfectly
                         # ended in a PLAN and the three statement CSVs were never opened.
                         # ⚠️ IT GOVERNS BOTH WRITERS SINCE 2026-09-06: §6's per-quarter
                         # upsert (`MERGE_EACH`) and §9's sweep. False makes §6 print
                         # each finished quarter's PLAN as it goes and write nothing —
                         # which is the honest reading of "apply", not a second knob.
                         # ⚠️ MEASURED ON HOSE_FPT, 2026-09-04: a 185-minute T4 round trip
                         #    over 71 filings accepted 128 of 213 statements, §9 planned
                         #    **96 WRITEs**, and **0** of them reached disk. Two knobs had
                         #    to be flipped by hand afterwards to finish a job the machine
                         #    had already done — which is `BND-1`'s loop wearing a second
                         #    face: the work is on disk, the CSV is not, and a green run
                         #    says nothing about which.
                         #   False -> PLAN ONLY. Right when you are about to REPAIR a row,
                         #            or want to read the refusals before spending disk.
                         # ⚠️ WHAT MAKES AN AUTOMATIC WRITE DEFENSIBLE IS THE REFUSALS, NOT
                         # THE EXTRA COMMAND (CLAUDE.md §6-2-quinquadragies, which made the
                         # LOCAL per-quarter merge automatic on the same argument). §9 passes
                         # `force_differs=False`, so a figure that DIFFERS from a good `pdf`
                         # row on disk is STILL refused however `OVERWRITE` is set — and the
                         # other three refusals stand untouched: an empty `sane` band, a
                         # cumulative income statement whose priors it cannot subtract, and
                         # a document any of whose layers RAISED. A backup of the three CSVs
                         # is taken by the first call that writes anything, and every changed
                         # cell is printed. `REPAIR` in §11 is still the only way past
                         # DIFFERS, and it is still opt-in and scoped.
                         # ⚠️ AND A DRY RUN UNDERSTATES A TWO-PASS WRITE, BY CONSTRUCTION:
                         # with nothing written, a later period is planned against the span
                         # the earlier one has not recorded yet, and reports the refusal it
                         # always would. That is a property of the dry run, not a result —
                         # which is the other reason False was the worse default: it could
                         # not even tell you what True would do.

# ⚠️ REPAIR — REPLACE A `pdf` ROW THAT IS ALREADY ON DISK AND WRONG. Name the exact
#    (quarter, statement) pairs; anything not named keeps the DIFFERS refusal. The quarter is
#    the REPO-NATIVE form here:   REPAIR = [("Q3-2019", "income_statement")]
# ⚠️ `OVERWRITE = True` IS THE WRONG TOOL FOR THIS, AND THE REASON IS MEASURED. It lifts DIFFERS
#    for every statement of every quarter in the run — and a `pdf_ocr_job` run is NOT the run
#    that wrote those rows: its `sane` band is rebuilt from disk where a full `build()`
#    accumulates one as it goes, so the two escalate DIFFERENTLY and the seeded run can win on
#    an EARLIER, POORER layer. On ACB 2026-08-30 it would have replaced a 33-item balance sheet
#    with a 19-item one while repairing another statement, reporting only "DIFFERS in N columns".
# ⚠️ Read the DIFFERS report in section 7 first, and decide against the FILING (a printed
#    subtotal, the next quarter's comparative column) — never by preferring the newer run.
REPAIR = []
REPAIR_APPLY = False     # False = print the plan and change nothing. True once you agree.

EXECUTE  = True          # False = resolve and print the plan, spend nothing
REHEARSE = True          # KAGGLE only: the worker side, locally, no quota (~60 s)

## 2 · Setup — validate the parameters, find the repo

In [ ]:
# ── SETUP — validate the parameters and find the repo ─────────────────────────
# ⚠️ Checked HERE, before a payload is built or a page is rendered: every one of these is a
# mistake that would otherwise surface hours later, or as a spent Kaggle round trip.
import os
import sys
from pathlib import Path

ENVIRONMENT = str(ENVIRONMENT).upper()
EXCHANGE = str(EXCHANGE).upper()
SYMBOL = str(SYMBOL).upper()
if ENVIRONMENT not in ("LOCAL", "KAGGLE"):
    raise ValueError(f"ENVIRONMENT must be 'LOCAL' or 'KAGGLE', not {ENVIRONMENT!r}")
if EXCHANGE not in ("HOSE", "HNX", "UPCOM"):
    raise ValueError(f"EXCHANGE must be HOSE, HNX or UPCOM, not {EXCHANGE!r}")

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "kaggle_gpu").is_dir()), None)
if REPO is None:
    raise RuntimeError(f"no src/kaggle_gpu at or above {Path.cwd()} — open this notebook "
                       f"from inside the repo.")
# ⚠️ `kgpu` stages the payload and talks to the Kaggle client relative to the CWD, so the
# notebook anchors itself the way a shell would. LOCAL does not need it and gets it anyway:
# one behaviour, printed, beats two that differ by a mode.
os.chdir(REPO / "src" / "kaggle_gpu")
for _p in (REPO / "src", REPO / "src" / "kaggle_gpu"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

# ⚠️ A LONG-LIVED KERNEL PINS THE REPO TO THE COMMIT IT FIRST IMPORTED — `import` is a no-op
# once a module is in `sys.modules`, so re-running this notebook after the repo moves underneath
# it runs the OLD code. The loud form is an AttributeError; ⚠️ the silent form is an OCR run
# executing a previous commit's parser while `metadata.json` records HEAD's hash — a run folder
# that names code it did not run. So the repo's OWN packages are dropped here and re-imported
# from disk on every pass; third-party ones (torch, onnxruntime) are left alone, they do not
# move. ⚠️ It re-imports, so run this notebook TOP TO BOTTOM.
_OURS = ("kgpu", "utils", "web_scraper")
_RELOADED = [_n for _n in list(sys.modules) if _n.split(".")[0] in _OURS]
for _n in _RELOADED:
    del sys.modules[_n]

from utils import progress                          # noqa: E402
from web_scraper import pdf_ocr_job as job          # noqa: E402

# ⚠️ FOLDED ONCE, HERE. "2026-04" and "2026-Q4" are one quarter, and the job name, the payload
# directory and the Kaggle kernel slug are all derived from this list — two spellings that
# reached those would be two runs racing for one slug. An EMPTY list folds to `None`, which is
# `plan()`'s own contract for "every quarter this ticker files".
# ⚠️ A LIST, AND NOTHING ELSE (2026-09-03). QUARTERS used to take the strings "ALL" and
# "OUTSTANDING" beside the list, and TWO TYPES IN ONE PARAMETER COST THREE MEASURED READINGS,
# every one of which reported the wrong mistake:
#   QUARTERS = ""          an empty string is FALSY, so it fell past the sentinel test into
#                          `canonical_quarters`, which reads empty as `None` — it opened EVERY
#                          quarter this ticker files, silently, and printed "ALL".
#   QUARTERS = "  "        strips to "" and falls the same way, except `canonical_quarters`
#                          then iterates the string CHARACTER BY CHARACTER: `' ' is not a
#                          quarter`, an error about the QUARTER FORM for a mistake in the MODE.
#   QUARTERS = "2014-Q4"   the brackets forgotten — refused with `must be "ALL" or
#                          "OUTSTANDING"`, an error about the MODE for a mistake in the LIST.
# The narrowing sentinel is `ONLY_MISSING` now, the list is only ever a list, and ONE message
# covers a string, a `None` and anything else that is not one.
if not isinstance(QUARTERS, (list, tuple)):
    raise TypeError(f'QUARTERS is a LIST of quarters — [] or ["2014-Q4"] — not {QUARTERS!r}. '
                    f"{job.QUARTER_FORM}. An EMPTY list is every quarter the ticker files, "
                    f"and ONLY_MISSING = True narrows it to the ones still `missing`.")
QUARTERS = job.canonical_quarters(QUARTERS)
# ⚠️ AN EMPTY LIST IS RESOLVED IN §3, NOT HERE, and it is the only thing that is: §3 is where
# the statement CSVs and the PDF index are read, and neither has been opened yet.
RESOLVE_FROM_DISK = QUARTERS is None
OUTSTANDING_ONLY = RESOLVE_FROM_DISK and ONLY_MISSING

# ⚠️ THE CASCADE IS PART OF A RUN'S PROVENANCE, NOT ONLY OF ITS COST (`TSS-1`). `tesseract@200`
# is layer 4 of 55 HERE and does not exist on a Kaggle worker, so the two machines run
# DIFFERENT cascades and a local re-parse of a T4-parsed ticker can win on a layer the row on
# disk never saw. `ONNX_ONLY` makes the two the same 53. It never overrides an explicit
# `LAYERS`, and the resolved list is recorded in the run folder either way.
from web_scraper.cafef_financials import FinancialsBuilder as _FB   # noqa: E402

if ONNX_ONLY and LAYERS is None:

    LAYERS = [_l.name for _l in _FB.LAYERS if _l.name.startswith("onnx")]

# ⚠️ TWO COMBINATIONS ARE REFUSED HERE RATHER THAN DISCOVERED AFTERWARDS, and both were
# measured on real runs:
#   (a) SPAN_OPERANDS needs OVERWRITE. A span operand is by definition a quarter already
#       reading `pdf`, so at OVERWRITE=False it is dropped before any OCR and the Q4 it
#       exists to unblock stays unwritable — the run would look complete and change nothing.
#   (b) OVERWRITE + MERGE_INTO_CSV passes `force_differs=True` into the automatic per-quarter
#       merge, i.e. it lifts DIFFERS for EVERY statement of every quarter in the run. On ACB
#       (2026-08-30) that would have replaced a 33-item balance sheet with a 19-item one while
#       repairing a different statement. §9's merge is unforced; REPAIR is the scoped escape.
if SPAN_OPERANDS and not OVERWRITE:
    raise ValueError("SPAN_OPERANDS needs OVERWRITE = True — a span operand is a quarter "
                     "already reading `pdf`, and OVERWRITE=False drops it before any OCR.")
if OVERWRITE and MERGE_INTO_CSV:
    raise ValueError("OVERWRITE = True passes force_differs into the automatic merge, which "
                     "lifts DIFFERS for every statement of the run. Leave MERGE_INTO_CSV off "
                     "and use §9 (unforced, one period at a time), or REPAIR for one row.")

# The task label EVERY progress line in this notebook carries, so it is kept SHORT: it is
# repeated on every row of every table below, and a 40-character label pushes a verdict table
# off the screen to say something §4 already printed. Two quarters or fewer are named; more
# are a count.
# ⚠️ Rebuilt in §3 when the sentinel resolves: a label reading "all quarters" over a
# three-quarter run is a progress line lying about its own denominator.
LABEL = f"{EXCHANGE}_{SYMBOL}" + (
    " " + " ".join(QUARTERS) if QUARTERS and len(QUARTERS) <= 2
    else f" {len(QUARTERS)}q" if QUARTERS else "")

# ⚠️ **ONE PLAN FOR THE WHOLE NOTEBOOK, AND THEREFORE ONE PERCENTAGE.** Every line printed
# from here down leads with `xx.x%` of THE WHOLE SESSION — not of the cell you are in — in the
# one shape `utils.progress` formats and nothing else writes:
#     ` 33.7% - step 5/15 HOSE_CTG 70q - wait kernel - [ 1.5 min] RUNNING`
# Before 2026-09-04 only §5 and §6 reported at all, each with a plan of its own, so a reader
# got `33.7%` from the run cell and bare prose from the nine cells around it and had no way to
# tell a session 3 % in from one 96 % in. The three honest denominators are still named, in
# the segments (`step 5/15`, `doc 2/3`, `page 40/96`).
# ⚠️ THE OCR STEPS ARE THE ROUND TRIP'S OWN (`kgpu.runner.RUN_STAGES`) ON KAGGLE, EMBEDDED
#    HERE RATHER THAN RUN AS A SECOND PLAN — `runner.run` looks its stages up BY KEY, so
#    handing it this plan makes its six steps six steps of this notebook and keeps ONE number
#    on the line. `final=False` is what stops its closing `done()` reading as "the notebook is
#    finished" and parking every cell after it at 100 %.
if ENVIRONMENT == "KAGGLE":
    from kgpu import runner as _runner              # noqa: E402

    _OCR_STAGES = list(_runner.RUN_STAGES)          # export upload push wait download merge
else:
    # ⚠️ Weighted 100 to match `RUN_STAGES`' own total, so the OCR is the same share of the
    # notebook on both machines and the two runs' percentages mean the same thing.
    _OCR_STAGES = [("parse", "OCR the filings", 100.0)]
OCR_KEYS = [_s[0] for _s in _OCR_STAGES]
NOTEBOOK_PLAN = [
    ("setup",    "setup",                1.0),
    ("gap",      "what is left",         1.0),
    ("job",      "resolve the job",      2.0),
    ("rehearse", "rehearse worker",      3.0),
    *_OCR_STAGES,
    ("results",  "read the run folders", 1.0),
    ("refused",  "refused vs written",   1.0),
    ("upsert",   "merge into the CSVs",  5.0),
    ("landed",   "did it land",          1.0),
    ("repair",   "repair one row",       1.0),
]
# ⚠️ THE WEIGHTS ARE NOMINAL AND SAY SO. They put the OCR where it belongs — ~86 % of the
# plan — and they measure no run: a filing accepted at layer 1 is ~1 min and one that defeats
# the cascade was 33 (§6-2-noviesdecies). A weight pretending to be measured would be §5
# rule 2 wearing a progress bar.
# ⚠️ RE-RUN §2 AFTER EDITING §1: the plan's SHAPE depends on ENVIRONMENT. The number is
#    monotone by construction, so re-running a cell out of order re-prints its step at the
#    percentage already reached rather than winding the bar back.
NB = progress.Stages(NOTEBOOK_PLAN, label=LABEL, final=False)
NB.begin("setup", f"{ENVIRONMENT} — parameters validated, repo found")
with NB.capture(nested=True):
    print(f"environment : {ENVIRONMENT}")
    print(f"ticker      : {EXCHANGE}_{SYMBOL}")
    print("quarters    : " + ("OUTSTANDING — resolved in §3 from what is on disk"
                              if OUTSTANDING_ONLY else
                              f"{QUARTERS or 'ALL — every quarter this ticker files'}"))
    # ⚠️ A KNOB THAT IS NOT READ HAS TO SAY SO WHERE IT WOULD BE READ. Silence is what lets
    # a reader believe a flag they set had an effect, and this one is inert the moment the
    # list names its own quarters.
    if QUARTERS and ONLY_MISSING:
        print("            : ⚠️ ONLY_MISSING is IGNORED — it is read only when QUARTERS is "
              "empty, and this run names its quarters.")
    print(f"overwrite   : {OVERWRITE}"
          + ("" if OVERWRITE else "   (quarters already `pdf` in all three are skipped)"))
    print(f"upsert csv  : {MERGE_INTO_CSV}"
          + ("   per quarter, as each finishes" if MERGE_INTO_CSV and ENVIRONMENT == "LOCAL"
             else "   after the pull" if MERGE_INTO_CSV else ""))
    # ⚠️ A KNOB THAT IS NOT READ HAS TO SAY SO WHERE IT WOULD BE READ — the same rule
    # `ONLY_MISSING` obeys two lines up. `MERGE_EACH` is `run_batch`'s argument and nothing
    # else's, so on KAGGLE (the worker cannot reach this disk) and on the one-process path
    # (that is `MERGE_INTO_CSV`) it is inert, and silence is what would let a reader believe
    # the CSVs were being written as the run went.
    print(f"merge each  : {MERGE_EACH}"
          + ("   each quarter is upserted the moment all three of its statements are in"
             if MERGE_EACH and ENVIRONMENT == "LOCAL" and ISOLATE_DOCUMENTS else
             "   ⚠️ IGNORED — read only on LOCAL + ISOLATE_DOCUMENTS; "
             + ("KAGGLE writes on the pull" if ENVIRONMENT == "KAGGLE"
                else "the one-process path is MERGE_INTO_CSV") if MERGE_EACH else
             "   nothing reaches the CSVs until §9"))
    print(f"bootstrap   : {FORCE_EMPTY_BAND}"
          + ("   an EMPTY `sane` band is written anyway — the only way a new ticker "
             "starts" if FORCE_EMPTY_BAND else "   an EMPTY `sane` band is REFUSED"))
    print(f"isolation   : "
          + ("one process per document, VRAM floor "
             f"{VRAM_FLOOR_MB} MiB   (`GPU-1`)" if ISOLATE_DOCUMENTS and ENVIRONMENT == "LOCAL"
             else "one process for the whole run" if ENVIRONMENT == "LOCAL" else "n/a — KAGGLE"))
    print(f"cascade     : "
          + (f"{len(LAYERS)} layer(s)" if LAYERS else "the full cascade")
          + ("   onnx only — the cascade a Kaggle worker runs (`TSS-1`)"
             if ONNX_ONLY and LAYERS else ""))
    # ⚠️ **WHAT THIS CASCADE CAN RECOVER, DERIVED FROM THE LAYERS THEMSELVES.** The methods are
    # SHARED CODE — `FinancialsBuilder.LAYERS` and the `ParseLayer` flags — not notebook
    # settings, so every ticker driven from here gets all of them and there is nothing to "turn
    # on". What the readout is for is the log: a line ending `[onnx@200+noteshead]` means a
    # widening rule won that statement, and this says which rules were even reachable.
    # ⚠️ Listed from `dataclasses.fields`, never from a hand-written list — a gloss typed here
    # would be a second copy of the cascade and would be wrong the first time a flag is added.
    # `ParseLayer`'s docstring is where each one is explained and measured.
    # ⚠️ **`is_strict` IS THE LINE THAT MATTERS**: a layer reading the page AS PRINTED must
    # never run after one that widens what may be believed, so the strict reads come first and
    # a widening layer only ever judges a statement all of them refused.
    import dataclasses                                    # noqa: E402
    import textwrap                                       # noqa: E402

    from web_scraper.cafef_financials import ParseLayer   # noqa: E402

    _CASCADE = [l for l in _FB.LAYERS if LAYERS is None or l.name in set(LAYERS)]
    _WIDE = sorted(f.name for f in dataclasses.fields(ParseLayer)
                   if any(getattr(l, f.name) is True for l in _CASCADE))
    _STRICT = sum(1 for l in _CASCADE if l.is_strict)
    print(f"recoveries  : {_STRICT} strict read(s), then {len(_CASCADE) - _STRICT} widening "
          f"layer(s) carrying {len(_WIDE)} flag(s)")
    print(textwrap.fill(" ".join(_WIDE), 92, initial_indent="              ",
                        subsequent_indent="              "))
    print(f"repo        : {REPO}")
    print(f"cwd         : {Path.cwd()}")
    print(f"code        : {REPO / 'src'}"
          + (f"   ({len(_RELOADED)} cached module(s) dropped, re-imported from disk)"
             if _RELOADED else "   (first import in this kernel)"))
    # ⚠️ The percentage is a POSITION IN THE PLAN and not a fraction of the time left — a filing
    # accepted at layer 1 of 47 costs ~1 min and one that defeats the cascade cost 33. Said here,
    # once, because it is on every line below it.
    print(f"log shape   : {progress.format_line(0.337, 'task', 'sub-task', 'detail')}")
    print(f"              overall % of THIS NOTEBOOK — {len(NOTEBOOK_PLAN)} steps, the OCR worth "
          f"{100 * sum(_s[2] for _s in _OCR_STAGES) / sum(_s[2] for _s in NOTEBOOK_PLAN):.0f}%.")
    print("              A position in the plan, never a fraction of the time — a LOWER bound, "
          "so a run finishes early rather than stalling at 99 %.")
NB.end()

## 3 · What is left — the gap on disk, and what a re-run cannot change

In [ ]:
# ── WHAT IS LEFT — the gap on disk, and what a re-run cannot change ───────────
# ⚠️ THE QUESTION THIS ANSWERS IS THE ONE THAT DECIDES `QUARTERS`, and until 2026-09-02 the
# notebook could not answer it: you had to know which (quarter, statement) cells of this ticker
# still read `missing`, and the only way to find out was an ad-hoc script over the three CSVs.
#
# ⚠️ IT IS NOT A SECOND RULE. The quarters come from `documents()` through `job.plan()` — the
# same call the run makes — "already done" is `job.parsed_reports()`, which is `pdf` and nothing
# else, and a cell a past run PROVED unproducible is dropped by `settled_absences`.
#
# ⚠️ `use_data_root()` FIRST, AND IT IS LOAD-BEARING (`CWD-1`). `fin.STATEMENTS_DIR` is a
# RELATIVE default read at call time, and §2 has just `os.chdir`-ed into `src/kaggle_gpu` — so
# without this every quarter reads `absent`, which is a legitimate state for a ticker being
# bootstrapped and therefore looks like nothing is wrong.
NB.begin("gap", "the three statement CSVs and the PDF index — no OCR")
with NB.capture(nested=True):
    from web_scraper import cafef_financials as fin      # noqa: E402
    from web_scraper import pdf_ocr_batch                # noqa: E402

    job.use_data_root(REPO / "raw_data" / "cafef")
    _builder = fin.FinancialsBuilder(logger=None)

    # ⚠️ RESOLVED, NEVER DEFAULTED — and how it resolved is printed, because "read off
    # templates.csv" and "fingerprinted over the network" are not the same claim (`TPX-1`).
    [PLAN] = pdf_ocr_batch.plan_batch(
        [SYMBOL], exchange=EXCHANGE, reports_root=REPO / "reports" / "pdf_ocr",
        allow_parent=ALLOW_PARENT, span_operands=SPAN_OPERANDS, template=TEMPLATE,
        builder=_builder)
    TEMPLATE, TEMPLATE_HOW = PLAN.template, PLAN.template_how

    print(f"{PLAN.key}   template {TEMPLATE} ({TEMPLATE_HOW})   "
          f"{PLAN.filed} quarter(s) filed, {PLAN.complete} complete")
    print("")
    # ⚠️ NO FILINGS AND NOTHING OUTSTANDING PRINT THE SAME LINE OTHERWISE, and they are opposite
    # answers: one says the ticker is done, the other that nothing was ever measured (§5 rule 2).
    if not PLAN.filed:
        print("  ⚠️ this ticker files NO document `documents()` will open — an absent PDF "
              "index, or")
        print("     everything before FINANCIALS_PERIOD_MIN. Nothing here says the ticker "
              "is done.")
    elif not PLAN.quarters and not PLAN.settled:
        print("  every filed quarter reads `pdf` in all three statements. Nothing is outstanding.")
    else:
        for _q in PLAN.quarters:
            _tag = "SPAN OPERAND — re-parsed only to record `months`" if _q in PLAN.operands else \
                   "open — a re-run could still win it"
            print(f"  {_q:9} {_tag}")
        for _q, _reports in sorted(PLAN.settled.items()):
            for _r in _reports:
                print(f"  {_q:9} {_r:18} SETTLED — the filing contains no such statement")
        print("")
        print(f"  {len(PLAN.quarters)} quarter(s) with an OPEN cell "
              f"(of which {len(PLAN.operands)} are span operands), "
              f"{sum(len(v) for v in PLAN.settled.values())} SETTLED cell(s)")

    # ⚠️ A SETTLED CELL IS `missing` FOREVER, and re-running it costs the full cascade to
    # return the same word. ACB's Q2-2009 and Q3-2009 cash flows were put through all 50
    # layers FOUR times on
    # 2026-08-30 before anything recorded why: both filings are three-page `BÁO CÁO TÀI CHÍNH TÓM
    # TẮT` forms (Mẫu CBTT-03) with no cash flow statement in them at all.
    # ⚠️ AND AN EMPTY SETTLED SET IS SILENCE, NOT A CLEAN BILL: a run older than artefact schema v4
    # recorded no reason, so a cell reading "open" here may still be unwinnable and merely
    # unmeasured (§5 rule 2).
    if PLAN.settled:
        print("")
        print("  `missing` is the correct and PERMANENT answer for the SETTLED rows (§5 rule 24).")

    # ⚠️ WHICH QUARTERS THE RUN ACTUALLY TAKES — three modes, and an EMPTY `QUARTERS` is resolved
    # HERE and nowhere else. An ONLY_MISSING that resolves to nothing RAISES rather than falling
    # through: `plan()` reads an empty `quarters` as "every quarter this ticker files", so the one
    # thing a "nothing left to do" answer must not do is silently open 70 filings.
    if OUTSTANDING_ONLY:
        if not PLAN.quarters:
            raise RuntimeError(
                f"ONLY_MISSING resolved to nothing for {PLAN.key}: every filed quarter either "
                f"reads `pdf` in all three statements or is SETTLED. Name the quarters "
                f"explicitly, or set ONLY_MISSING = False, if you meant to re-parse "
                f"something anyway.")
        QUARTERS = PLAN.quarters
    elif RESOLVE_FROM_DISK:
        # ⚠️ EVERY QUARTER THE TICKER FILES — including the ones already `pdf`, which is the point:
        # this is the mode that gets a ticker to FULL coverage rather than filling its gaps. It
        # needs OVERWRITE (validated in §2) and, on this card, ISOLATE_DOCUMENTS.
        QUARTERS = [job.as_quarter(t.period) for t in
                    job.plan(_builder, EXCHANGE, SYMBOL, allow_parent=ALLOW_PARENT,
                             template=TEMPLATE)]
        PLAN.quarters = QUARTERS
    else:
        # ⚠️ `else`, not `elif QUARTERS`: an empty list is the two branches above, so everything
        # reaching here NAMES its quarters. A truthiness test would leave a fourth, silent
        # path that ran with `PLAN.quarters` still holding §3's outstanding set — a run
        # taking quarters nobody asked for, with nothing printing a difference.
        PLAN.quarters = list(QUARTERS)

    # ⚠️ The label reaches EVERY line below, so it is a count once past two quarters — §4 prints
    # the list, and repeating it on each row of a 70-quarter verdict table buys nothing.
    LABEL = f"{EXCHANGE}_{SYMBOL}" + (" " + " ".join(PLAN.quarters)
                                      if 1 <= len(PLAN.quarters) <= 2
                                      else f" {len(PLAN.quarters)}q")
    NB.task = LABEL          # the sentinel has resolved; the denominator on the line is now true
    print("")
    _MODE = "OUTSTANDING" if OUTSTANDING_ONLY else "ALL" if RESOLVE_FROM_DISK else "explicit"
    print(f'  QUARTERS = {_MODE} -> {len(PLAN.quarters)} document(s)'
          + (f": {' '.join(PLAN.quarters)}" if len(PLAN.quarters) <= 12 else
             f": {' '.join(PLAN.quarters[:6])} … {' '.join(PLAN.quarters[-3:])}"))

    # ⚠️ **THE THREE CSVs THEMSELVES, BECAUSE `MERGE_EACH` WRITES INTO THEM AS THE RUN GOES —
    # and because a MISSING file is not the obstacle a reader expects it to be.**
    # `FinancialsBuilder._write` creates the directory and the file, so "there is no CSV yet"
    # costs nothing by itself. What costs is what a missing CSV IMPLIES: no `pdf` row on disk,
    # so `seed_history` reconstructs no magnitude band, so `sane` fails open, so every
    # statement is refused, so there is still no CSV — `BND-1`, and it is a loop that only
    # FORCE_EMPTY_BAND breaks. Printed HERE, where nothing has been spent, because the
    # alternative is learning it after a whole-ticker parse (HOSE_FPT, 2026-09-04).
    # ⚠️ `_builder._existing` is `plan_merge`'s own reader, not a second one — a count taken
    # by a different reader here could disagree with the merge that follows it.
    print("")
    CSV_ON_DISK = {}
    for _report in fin.REPORTS:
        _rows = _builder._existing(EXCHANGE, SYMBOL, TEMPLATE, _report)
        CSV_ON_DISK[_report] = sum(1 for _r in _rows.values() if _r.get("source") == "pdf")
        _path = Path(fin.statement_path(TEMPLATE, _report, EXCHANGE, SYMBOL))
        print(f"  {_report:18} "
              + (f"{len(_rows):>3} row(s), {CSV_ON_DISK[_report]:>3} `pdf`   {_path.name}"
                 if _path.is_file() else
                 f"⚠️ NO FILE — {_path.name} is created by the first write that clears the "
                 f"refusals"))
    # ⚠️ **NO `pdf` ROW ANYWHERE IS THE BOOTSTRAP CASE, AND IT IS NOT THE SAME TEST AS "NO
    # FILE".** A CSV that exists holding only `missing`/`cafef` rows seeds no band either
    # (`seed_history` reads `pdf` and nothing else, §5 rule 24), so a file-existence test would
    # call such a ticker ready and every write would still be refused.
    NEEDS_BOOTSTRAP = not any(CSV_ON_DISK.values())
    if NEEDS_BOOTSTRAP:
        print("")
        print(f"  ⚠️ {PLAN.key} HAS NO `pdf` ROW ON DISK — this run BOOTSTRAPS the ticker, and")
        print("     `seed_history` has nothing to rebuild a magnitude band from, so `sane` "
              "FAILS OPEN")
        print("     on every statement of it. A quarter whose filing produces all three is "
              "written")
        print("     anyway (`BND-1` is a loop, not a guard — MERGE_EACH says why), and the "
              "three CSVs")
        print("     are CREATED by the first such write.")
        print("     ⚠️ THOSE ROWS PASS NO MAGNITUDE GUARD. Screen them by arithmetic — two "
              "statements")
        print("        agreeing on one figure, a printed subtotal closing — before quoting "
              "any of them.")
        print("        Each is printed as it is written, recorded in the run folder's `merge` "
              "block,")
        print("        and counted again in §10.")
NB.end()

## 4 · The plan — what would run, before anything is spent

In [ ]:
# ── THE JOB — resolved and printed, before anything is spent ──────────────────
# ⚠️ Both branches end at the SAME object. `pdf_ocr.job()` writes a `JobSpec`'s fields into the
# worker notebook's parameter cell, and the worker builds the JobSpec from them — so a LOCAL run
# and a KAGGLE run of the same parameters are one procedure on two machines, not two. What
# differs is the stack, and every run records its `stack_fingerprint`.
NB.begin("job", "resolve the spec, count the ceiling — nothing is spent")
with NB.capture(nested=True):
    SPEC = CFG = PREPARED = None

    if ENVIRONMENT == "LOCAL":
        SPEC = job.JobSpec(
            exchange=EXCHANGE, symbol=SYMBOL, periods=PERIODS, quarters=PLAN.quarters,
            allow_parent=ALLOW_PARENT, overwrite=OVERWRITE, template=TEMPLATE, layers=LAYERS,
            compare_with_disk=COMPARE, merge_into_csv=MERGE_INTO_CSV,
            force_empty_band=FORCE_EMPTY_BAND,
            notes=NOTES or f"ENVIRONMENT=LOCAL overwrite={OVERWRITE}",
        )
        # ⚠️ `prepare()` resolves the data root, the models, the TEMPLATE and the document list and
        # RAISES on any of them — no OCR, no PDF. It also raises, in as many words, when every
        # quarter you asked for is already parsed and OVERWRITE is False.
        PREPARED = SPEC.prepare()
        print("\n".join(PREPARED.describe()))
        print()
        for _t in PREPARED.tasks:
            print(f"  {_t.period:<8} {_t.file[:56]:<56} "
                  f"{os.path.getsize(_t.path) / 1024 ** 2:>6.1f} MB"
                  + ("  CUMULATIVE" if _t.cumulative else ""))
        # ⚠️ THE CEILING, BEFORE ANY OF IT IS SPENT. The bill is `pages x OCR passes`, and the 49
        # layers are only 7 passes — a layer that changes only the mapping or a gate re-maps
        # a parse the page cache already holds. Both numbers are free: `page_count` opens
        # the PDF without
        # rendering a pixel, and the pass count is a property of the cascade.
        # ⚠️ It is a CEILING, loose in the honest direction: `scan` stops as soon as all three
        # statements are behind it (BID Q3-2011 reads 7 pages of 32) and the cascade stops at the
        # first layer that accepts. What it tells you is which filing would be dear IF something in
        # it cannot be read — that is the only case that pays it.
        import fitz                                       # noqa: E402
        from web_scraper.cafef_financials import ocr_key  # noqa: E402

        PASSES = len({ocr_key(_l) for _l in PREPARED.layers})
        PAGES = 0
        for _t in PREPARED.tasks:
            try:
                with fitz.open(_t.path) as _d:
                    PAGES += _d.page_count
            except Exception as _e:                       # a damaged page tree is `scan`'s problem
                print(f"  ⚠️ could not count pages of {_t.file}: {_e}")
        print("")
        print(f"  ceiling      : {PAGES} page(s) x {PASSES} OCR pass(es) = "
              f"{PAGES * PASSES:,} page-reads at most")
        print(f"                 ~{PAGES * PASSES * 0.65 / 60:.0f} min at 0.65 s/page "
              f"(onnx@200 on this laptop; the 300/400 dpi passes cost more).")
        print("                 A filing accepted at layer 1 pays ONE pass over the pages "
              "up to its last")
        print("                 statement, which is the usual case — see the run log.")

        if PREPARED.template != "bank":
            print(f"\n⚠️ CRP-1: this is a `{PREPARED.template}` filing. `C_LIABILITIES` still "
                  f"misses on corp,\n   so the balance sheet reconciles on the TRIVIAL "
                  f"`assets == resources` — true by\n   construction on any page that reads both. "
                  f"Nothing from a non-bank run may be\n   quoted as a fundamental yet.")
    else:
        from kgpu import pdf_ocr, runner                 # noqa: E402

        CFG = pdf_ocr.job(
            SYMBOL, exchange=EXCHANGE, periods=PERIODS, quarters=PLAN.quarters,
            allow_parent=ALLOW_PARENT, overwrite=OVERWRITE, template=TEMPLATE, layers=LAYERS,
            compare=COMPARE, notes=NOTES, merge_statements=MERGE_INTO_CSV,
            # ⚠️ NOT a worker parameter. The worker cannot upsert — it writes /kaggle/working and
            # exits — so this is the PULL's knob, read by `runner.merge_statements` on this
            # machine.
            force_empty_band=FORCE_EMPTY_BAND,
        )
        print("\n".join(pdf_ocr.describe(CFG)))
        print()
        # The filings this selects are the filings the WORKER will open: `plan()` runs HERE, so the
        # payload cannot diverge from the worker's own choice.
        runner.plan(CFG)
NB.end()

## 5 · Rehearse — KAGGLE only: the worker side, locally, no quota

In [ ]:
# ── STAGE + REHEARSE — KAGGLE only: the worker side, locally, no quota ────────
# ⚠️ THE PAYLOAD IS STAGED HERE, AND IT HAS TO BE: a rehearsal runs the worker against
# `.payload/<job>/`, so there is nothing to rehearse until that exists. `export` is local and
# free — it writes the zip, it does not upload; the RUN cell below re-exports and uploads, so
# nothing here commits you to anything.
# ⚠️ The rehearsal runs no OCR pass. What it proves is that the payload holds every input the
# parse reads, under BOTH of Kaggle's mount layouts, and it prints the magnitude band `sane`
# will get. AN EMPTY BAND IS THE WARNING TO STOP FOR: `sane` fails open without one, and that
# is the documented way a run writes a wrong figure (CLAUDE.md §6-2-octodecies).
# ⚠️ ONE STEP OF THE NOTEBOOK'S OWN PLAN, NOT A SECOND PLAN. This cell used to build a
# `Stages` of its own and print `50.0% - step 1/2 …` beside a run cell printing its own
# percentage of a different denominator — two bars, neither answering "how far through the
# whole thing am I?". `inside()` moves through THIS step instead, and `capture()` re-emits
# `export`'s and `rehearse`'s own output as the DETAIL of it.
if ENVIRONMENT == "KAGGLE" and REHEARSE:
    from kgpu import export, runner               # noqa: E402

    NB.begin("rehearse", "stage the payload — local, no upload, no quota")
    with NB.capture(nested=True):
        export.export(CFG)                        # -> .payload/<job>/  (no upload)
    NB.inside(0.5, "rehearse the worker — both Kaggle mount layouts")
    with NB.capture(nested=True):
        runner.rehearse(CFG)
    NB.end("rehearsed — nothing was spent")
else:
    # ⚠️ A SKIPPED STEP CLAIMS ITS WEIGHT rather than redistributing it: the plan is the plan,
    # and "we did not have to do that" is progress through it.
    NB.skip("rehearse", "REHEARSE = False" if ENVIRONMENT == "KAGGLE"
            else "LOCAL — nothing to rehearse")


## 6 · Run

In [ ]:
# ── RUN ───────────────────────────────────────────────────────────────────────
# ⚠️ One line shape on both machines — ` 33.7% - <task> - <sub-task> - <detail>`, one formatter
# (`utils.progress`), so the two cannot drift. LOCAL the task is the DOCUMENT and the sub-task
# its position in the cascade; KAGGLE the task is the STEP of the round trip.
# ⚠️ THE OVERALL % IS A POSITION IN THE PLAN, NOT A FRACTION OF THE TIME. A filing accepted at
# its FIRST OCR pass is ~1 min and one that defeats all 24 of them was 33, so the number is a
# LOWER BOUND — a run finishes early, it does not stall at 99 %.
#
# ⚠️ **`ISOLATE_DOCUMENTS` IS WHAT MAKES A WHOLE-TICKER RUN POSSIBLE ON THIS CARD.** Measured
# 2026-09-02: an 18-document run inside ONE process cleared three filings and then every
# `onnx@*` layer raised `CUDA failure 2: out of memory` — 294 of them — and the cascade went on
# and reported `pdf` for statements it had been unable to read. `pdf_ocr_batch.run_batch` spawns
# one process per document and waits for the card to have `VRAM_FLOOR_MB` free before each; the
# same 25 documents then ran with **0 engine errors**. It changes no semantics, because
# `seed_history` re-seeds `sane` from DISK per document and `PdfParser._ocr_cache` is scoped to
# one filing — see the module docstring.
FOLDERS: list = []
LATEST = EXIT = None

if not EXECUTE:
    # ⚠️ SKIPPED BY NAME, one line each, rather than one jump to the last OCR step. `skip`
    # advances to a stage's CEILING, so skipping the last would claim all six on a line
    # reading "merge into repo" — a step nobody asked about, credited for work nobody did.
    for _k in OCR_KEYS:
        NB.skip(_k, "EXECUTE = False — the plan above is resolved and nothing was spent")
elif ENVIRONMENT == "LOCAL" and ISOLATE_DOCUMENTS:
    from web_scraper import pdf_ocr_batch                # noqa: E402

    # ⚠️ **A BOOTSTRAP IS SAID BEFORE IT HAPPENS, NOT REFUSED.** Until 2026-09-06 this
    # raised here — a ticker with no `pdf` row has no magnitude band, so every write was
    # refused, and the run would have parsed every filing and landed none of them. The write
    # now happens for any quarter whose filing produced all three statements, so what is left
    # to do is name what that costs, once, where the hours are about to be spent.
    NB.begin("parse", f"{len(PLAN.quarters)} document(s), one process each, "
                      f"VRAM floor {VRAM_FLOOR_MB} MiB"
                      + ("   upserting each quarter as its three statements land"
                         if MERGE_EACH else ""))
    # ⚠️ AFTER `begin`, so the line carries the PARSE stage's label and its percentage — a
    # warning printed at the previous stage's position reads as being about that stage.
    if NEEDS_BOOTSTRAP and MERGE_EACH:
        NB.note(f"{PLAN.key} has no `pdf` row on disk — the three CSVs are CREATED by the "
                f"first quarter that produces all three statements, and every row this run "
                f"writes passes NO magnitude guard (`sane` fails open with no band, `BND-1`). "
                f"Screen them by arithmetic before quoting any of them.")
    # ⚠️ `progress=NB` is what stops the bar standing still through the longest thing the
    # notebook does: `run_batch` moves it ONE DOCUMENT AT A TIME through this step, and its own
    # lines come out as its detail. ⚠️ Each document is a SUBPROCESS that inherits stdout, so
    # its per-page lines go to the kernel log rather than into this cell — they are in that
    # document's own `run.log`, in the same shape.
    # ⚠️ **`merge_each` IS THE ONLY THING HERE THAT REACHES `raw_data/`.** `force_differs` is
    # not in `run_batch`'s signature and cannot be passed from it, so THREE of the four
    # refusals stand exactly as they do in §9 — a cumulative income statement whose priors it
    # cannot subtract, a figure that DIFFERS from a good `pdf` row, and a document any of
    # whose layers RAISED. What changes is WHEN: a quarter is written between documents, so an
    # interrupted run keeps what it has read.
    # ⚠️ **THE FOURTH — AN EMPTY `sane` BAND — IS LIFTED, AND THERE IS NO `force_empty_band`
    # ARGUMENT LEFT TO DECIDE OTHERWISE.** This path only ever merges a quarter whose filing
    # produced all three statements, and such a quarter is written band or no band: refusing
    # it is `BND-1`'s loop rather than a guard (§1's MERGE_EACH has the reasoning). Every such
    # row is printed as it is written, recorded in the run folder as `band: 0`, and counted
    # again in §10 — and it passed NO magnitude guard. FORCE_EMPTY_BAND still reaches §9,
    # which is what sees the quarters this path HELD.
    FOLDERS = pdf_ocr_batch.run_batch(
        [PLAN], layers=LAYERS, allow_parent=ALLOW_PARENT, overwrite=OVERWRITE,
        compare=COMPARE, notes=NOTES or f"{EXCHANGE}_{SYMBOL} — one process per document",
        vram_floor_mb=VRAM_FLOOR_MB, merge_each=MERGE_EACH, merge_apply=MERGE_APPLY,
        merge_reports=MERGE_REPORTS, progress=NB)
    NB.end(f"{len(FOLDERS)} run folder(s)")
elif ENVIRONMENT == "LOCAL":
    # ⚠️ THE OLD PATH, AND IT IS KEPT FOR ONE DOCUMENT AT A TIME. `job.run` parses every planned
    # filing in THIS process, which is right for a repair of one quarter and is what died at
    # document 4 of 18. It prints the progress line itself and writes the SAME line into the run
    # folder's `run.log`, so what you read here is what a later reader gets.
    # ⚠️ `nested=True` because those lines ALREADY lead with a percentage — of that run's own
    # documents, not of this notebook. Printed verbatim they would put two numbers on one line
    # and the second would appear to walk backwards; split, the inner percentage is dropped and
    # its `layer 12/47 …` / `page 40/96 …` segments are kept.
    NB.begin("parse", "one process for the whole run — right for ONE document")
    with NB.capture(nested=True):
        LATEST = job.run(SPEC)
    FOLDERS = [LATEST]
    NB.end(LATEST.name)
else:
    from kgpu import runner                              # noqa: E402

    if MERGE_INTO_CSV:
        NB.note("MERGE_INTO_CSV is on: accepted statements are upserted into "
                "raw_data/.../statements/ after the pull, with a backup taken first "
                "and every changed cell printed.")
    # `refresh_data=True` re-exports and re-uploads the payload every time — correct, because
    # the filter above may have changed since the last run of this job.
    # ⚠️ THE NOTEBOOK'S OWN PLAN IS HANDED STRAIGHT TO `runner.run`, because six of its stages
    # ARE the round trip's (§2 embedded `RUN_STAGES` by key). So the round trip reports as
    # steps 5..10 of 15 and the reader keeps ONE number. `final=False` (§2) is why its closing
    # `done()` ends the round trip rather than the notebook.
    EXIT = runner.run(CFG, refresh_data=True, progress=NB)
    NB.note(f"exit {EXIT}   (0 = COMPLETE and pulled)")

## 7 · The result — verdicts from the run folder

In [ ]:
# ── READ THE RUN FOLDERS ────────────────────────────────────────────────
# ⚠️ Read back from disk rather than from anything in memory, so this measures what a later
# reader would actually get. `metadata.json` already carries the whole scorecard in `results`.
# ⚠️ **A BATCH IS MANY FOLDERS, ONE PER DOCUMENT.** `FOLDERS` comes from the run cell; when the
# kernel was restarted between the two, fall back to this ticker's folders newer than the
# newest CSV backup — never to "the newest folder" alone, which on a batch is the LAST document
# and would report a 70-quarter run as a one-quarter one.
NB.begin("results", "read back from disk, not from memory")
with NB.capture(nested=True):
    import json                                          # noqa: E402

    PATTERN = f"*__{EXCHANGE.lower()}_{SYMBOL.lower()}__pdf_ocr"
    if not FOLDERS:
        FOLDERS = sorted((REPO / "reports" / "pdf_ocr").glob(PATTERN), key=lambda p: p.name)[-1:]
    LATEST = FOLDERS[-1] if FOLDERS else None
    META = MERGE = None
    RESULTS: list = []

    if LATEST is None:
        print(f"no run folder matching {PATTERN}")
    else:
        META = json.loads((LATEST / "metadata.json").read_text(encoding="utf-8"))
        inputs, ocr = META.get("inputs", {}), META.get("environment", {}).get("ocr", {})
        SCHEMA = META.get("schema_version", 1)
        print(f"{len(FOLDERS)} run folder(s), {FOLDERS[0].name} … {LATEST.name}")
        print(f"  commit       : {META.get('git_commit')}")
        print(f"  template     : {inputs.get('template')}  ({inputs.get('template_how')})")
        # ⚠️ THE TWO OCR HALVES FAIL INDEPENDENTLY — detection is onnxruntime, recognition is torch
        # — so "the GPU was used" is two questions. `ORT-1` is a green run that was half on the CPU
        # because onnxruntime ADVERTISED a provider the session then could not create.
        print(f"  detection    : {(ocr.get('det_providers') or ['?'])[0]}"
              f"   (onnxruntime {ocr.get('onnxruntime')})")
        print(f"  recognition  : {ocr.get('recognizer_device')}")
        print(f"  stack        : {ocr.get('stack_fingerprint')}"
              + (f"   ⚠️ PIN VIOLATIONS: {ocr['pin_violations']}"
                 if ocr.get("pin_violations") else ""))

        # ⚠️ **A RUN WHOSE ONNX LAYERS RAISED REPORTS `pdf` WITH A REAL LAYER AND A REAL ITEM
        # COUNT.** The rows below look identical to a good run; what happened is that the layer
        # could not run, the cascade went on, and something later won BY DEFAULT — which on this
        # machine is `tesseract@200`, layer 4 of 55. Measured 2026-09-02 on HOSE_CTG: 85 layers
        # raised `CUDA failure 2: out of memory` and 30 of 33 statements were reported `pdf`.
        # ⚠️ `pdf_ocr_merge` refuses such a document whole (`VCR-1`), so nothing reaches disk — but
        # that is the LAST line of defence and it is silent about WHY until §8. This says it here,
        # where the verdict table is read.
        RAISED = {}
        for _folder in FOLDERS:
            for _doc in sorted((_folder / "documents").glob("*.json")):
                _d = json.loads(_doc.read_text(encoding="utf-8"))
                if _d.get("engine_errors"):
                    RAISED[_d.get("period", _doc.stem)] = _d["engine_errors"]
            _m = json.loads((_folder / "metadata.json").read_text(encoding="utf-8"))
            RESULTS += _m.get("results", [])
        if RAISED:
            print()
            print(f"  ⚠️ {len(RAISED)} document(s) had at least one layer RAISE rather than "
                  f"refuse.")
            print("     Whatever won them won BY DEFAULT, and the merge refuses them whole.")
            for _p in sorted(RAISED)[:8]:
                print(f"       {_p:10} {len(RAISED[_p])} layer(s): "
                      f"{', '.join(l for l, _ in RAISED[_p][:3])}")
            _kinds = sorted({str(w).split(";")[0].strip()[:70]
                             for e in RAISED.values() for _l, w in e})
            for _k in _kinds[:3]:
                print(f"       cause: {_k}")
            print("     ⚠️ `out of memory` means the card was short — raise VRAM_FLOOR_MB, close "
                  "other")
            print("        CUDA processes, and re-run those quarters. Nothing of theirs is "
                  "on disk.")


        # ⚠️ **WHICH FILING EACH STATEMENT ACTUALLY CAME FROM (`ALT-1`).** `documents()` returns
        # ONE document per period and a quarter can have several, so a statement every layer
        # refused on the chosen filing is retried on the others of the same period and ENTITY.
        # When that succeeds the row on disk names THAT filing, not the one the document block
        # above names — and if this cell did not print it, nothing a reader sees would.
        # ⚠️ Measured on TCB Q2-2019: its closing balance is printed under the company's round
        # stamp in the AUDITED filing and no engine, DPI or crop reads it, while the REVIEWED
        # filing of the same quarter reads the whole tail cleanly at layer 1.
        ALT = {}
        for _folder in FOLDERS:
            for _doc in sorted((_folder / "documents").glob("*.json")):
                _d = json.loads(_doc.read_text(encoding="utf-8"))
                for _rep, _got in (_d.get("accepted") or {}).items():
                    if _got.get("document"):
                        ALT[(_d["period"], _rep)] = (_got["document"],
                                                     _got.get("assurance", ""))
        if ALT:
            print()
            print(f"  ⚠️ {len(ALT)} statement(s) came from a DIFFERENT filing of the same "
                  f"period and entity:")
            for (_p, _rep), (_file, _ass) in sorted(ALT.items()):
                print(f"       {_p:10} {_rep:18} {_ass:10} {_file}")
            print("     ⚠️ The ENTITY is fixed by `alternates`, so none of these changed which "
                  "company the")
            print("     row describes; the ASSURANCE may be lower, and that is the trade.")
        print()
        print(f"  {'period':10} {'report':18} {'layer':30} {'items':>5}  {'status':8} verdict")
        for r in sorted(RESULTS, key=lambda r: (fin._period_key(r["period"]), r["report"])):
            print(f"  {r['period']:10} {r['report']:18} {(r['layer'] or '—'):30} "
                  f"{r['items']:>5}  {r['status']:8} {r['verdict']}")
        # ⚠️ `seconds` is the DOCUMENT's cost repeated on each of its three report rows, so it is
        # summed per PERIOD. A set would also collapse two documents that took the same time.
        PER_DOC = {r["period"]: r["seconds"] for r in RESULTS}
        _ok = sum(1 for r in RESULTS if r["status"] == "pdf")
        print(f"\n  parse: {sum(PER_DOC.values()) / 60:.1f} min over {len(PER_DOC)} document(s)"
              f"   {_ok} of {len(RESULTS)} statement(s) accepted")
NB.end()

## 8 · Refused vs written — two questions, two places

In [ ]:
# ── WHAT WAS REFUSED, AND WHAT WAS WRITTEN ───────────────────────────────
#   the PARSE refused a statement   -> the document JSON's `absent_reasons`, and `run.log`
#   the MERGE refused a statement   -> the `merge` block, written by whatever ran the UPSERT
# ⚠️ ON KAGGLE THOSE ARE TWO MACHINES. A cell that greps the worker's `run.log` for
# `WRITE `/`skip ` finds nothing on a Kaggle run and, finding nothing, used to print "no
# refusals — every statement was accepted". That false success was printed over a run that
# wrote 0 of 201 accepted cells (HOSE_CTG, 2026-08-30).
#
# ⚠️ **THE REASON IS DATA NOW, NOT PROSE** (`absent_reasons`, artefact schema v4) — and since
# 2026-09-02 so are the ROWS behind it (`absent_rows`). A reason names the SYMPTOM (`no total
# assets`); the rows say WHAT THE FILING PRINTS where the chart expects that anchor, which is
# the only thing a fix can be written from. Recovering that used to cost a second OCR run.
NB.begin("refused", "the PARSE refused, and what the MERGE decided")
with NB.capture(nested=True):
    if FOLDERS:
        print("── the PARSE refused ────────────────────────────────────────")
        ABSENT: dict = {}
        for _folder in FOLDERS:
            for _doc in sorted((_folder / "documents").glob("*.json")):
                _d = json.loads(_doc.read_text(encoding="utf-8"))
                for _rep, _tried in (_d.get("absent_reasons") or {}).items():
                    ABSENT[(_d["period"], _rep)] = (
                        _tried, (_d.get("absent_rows") or {}).get(_rep))
        if not ABSENT:
            print("  nothing — every statement the cascade opened was accepted")
        for (_period, _rep), (_tried, _rows) in sorted(
                ABSENT.items(), key=lambda kv: (fin._period_key(kv[0][0]), kv[0][1])):
            print(f"  {_period:9} {_rep:18}")
            for _layer, _why in _tried:
                print(f"      [{_layer:28}] {_why}")
            # ⚠️ THE ROWS ARE THE CAUSE AND THE REASON IS THE SYMPTOM. Printed only for the
            # statements this run could not accept, and only the EARLIEST reading of them — the
            # last layer is always the most relaxed one and its rows answer a question nobody
            # asked (§6-2-duovicies' trap for the reason applies to the rows too).
            if _rows and SHOW_ABSENT_ROWS:
                print(f"      rows read at [{_rows['layer']}], pages {_rows['pages']}, "
                      f"{len(_rows['rows'])} row(s) — the ones naming a TOTAL:")
                for _r in _rows["rows"]:
                    _lab = (_r["label"] or "").upper()
                    if any(w in _lab for w in ("TỔNG", "TONG", "CUỐI", "CUOI", "ĐẦU", "DAU")):
                        print(f"        {_r['key'][:54]:54} {_r['values'][:2]}")
                        print(f"          {_r['label'][:96]}")

        # ⚠️ NOT a refusal — a fact about the FILING. A page whose scan is turned reads as vertical
        # noise, and before 2026-08-30 that cost a whole statement in silence (BID Q3-2011).
        TURNED = [ln for _f in FOLDERS
                  for ln in (_f / "run.log").read_text(encoding="utf-8",
                                                       errors="replace").splitlines()
                  if "text lines are vertical" in ln]
        if TURNED:
            print("\n── pages the READ had to turn ──────────────────────────────")
            for ln in TURNED[:12]:
                print("  " + progress.detail_of(ln))

        print("\n── the MERGE decided ───────────────────────────────────────")
        EVENTS = []
        for _folder in FOLDERS:
            _m = json.loads((_folder / "metadata.json").read_text(encoding="utf-8"))
            for _ev in (_m.get("merge") or {}).get("events", []):
                EVENTS += [(d, _ev["applied"]) for d in _ev["decisions"]]
        if EVENTS:
            for _d, _applied in sorted(EVENTS, key=lambda e: (fin._period_key(e[0]["period"]),
                                                              e[0]["report"])):
                mark = "WRITE " if _d["action"] == "write" else "skip  "
                items = f"[{_d['layer']}] {_d['items']} items" if _d["layer"] else ""
                print(f"  {mark} {_d['period']:9} {_d['report']:18} {items:32} {_d['reason']}")
                # ⚠️ A CAVEAT ON A WRITE IS LOUDER THAN A REFUSAL, because a refusal stops and a
                # write does not. Printed only for a WRITE: refusal 1 sets the note before
                # refusals 2-4 have had their say, so beside `skip` it would contradict the line.
                if _d.get("note") and _d["action"] == "write":
                    print(f"           ⚠️  {_d['note']}")
            _w = sum(1 for d, a in EVENTS if d["action"] == "write" and a)
            print(f"\n  -> {_w} statement(s) written, {len(EVENTS) - _w} refused or planned only")
            if not _w:
                print("  ⚠️ NOTHING REACHED raw_data/. On a ticker with no CSV yet the commonest "
                      "reason is an")
                print("     EMPTY `sane` band — set FORCE_EMPTY_BAND = True. It lifts ONE guard "
                      "and no other,")
                print("     so screen the artefact before quoting anything (`BND-1`).")
        else:
            print("  ⚠️ NO MERGE RAN against these run folders — the statement CSVs were not "
                  "opened.")
            print("     §9 below does it: MERGE_TWO_PASS = True, then MERGE_APPLY = True.")
NB.end()

## 9 · The merge — one period at a time, oldest first, and UNFORCED

In [ ]:
# ── THE MERGE — one period at a time, oldest first, and UNFORCED ──────────
# ⚠️ WHY NOT ONE CALL OVER THE FOLDER: `merge_run` runs `plan_merge` against disk FIRST and
# `_write` afterwards, so every decision in one call is taken against the SAME disk state.
# `_quarter_priors` reads a prior's `months` from that state, so the span a Q3 records reaches
# Q4's planner only in the NEXT call. That is `SPN-1`'s dependency, and it is why a batch that
# re-parses a span operand AND the Q4 it unblocks must merge them separately, oldest first.
# ⚠️ AND NOTHING HERE LIFTS A REFUSAL. `force_differs` is not passed, so a reading that
# disagrees with disk is refused exactly as it would be by default — which is the honest
# outcome, not a failure of this cell. `REPAIR` in §11 is the scoped escape.
# ⚠️ ONE BACKUP PER TICKER, taken by the first call that actually writes.
NB.begin("upsert", f"apply={MERGE_APPLY}   one period at a time, oldest first")
with NB.capture(nested=True):
    from web_scraper import pdf_ocr_batch                  # noqa: E402

    if not MERGE_TWO_PASS:
        print("MERGE_TWO_PASS = False — nothing was merged.")
    elif not FOLDERS:
        print("no run folder to merge — run the cells above first.")
    else:
        # ⚠️ `force_empty_band` IS NOT THE WHOLE ANSWER ANY MORE, so printing it alone
        # would understate what this pass will write. `merge_batch` lifts the band refusal
        # for any quarter whose filing produced ALL THREE statements — the same rule §6's
        # per-quarter write applies, so the two writers cannot disagree about one quarter —
        # and this flag governs only what that gate does not cover.
        print(f"       force_differs=False   force_empty_band={FORCE_EMPTY_BAND}"
              "   (+ always, for a quarter with all three statements)")
        print(f"       reports={MERGE_REPORTS or 'all three'}")
        # ⚠️ WHAT THIS PASS IS FOR ONCE §6 HAS ALREADY WRITTEN. Not a second write: every
        # quarter §6 landed is re-planned against disk and comes back `identical to the row
        # already on disk`, which is a CHECK. What it picks up is what §6 HELD — a filing that
        # produced two statements of three, a document whose layers RAISED — and any quarter
        # whose span operand only landed later in the run.
        if MERGE_EACH and ENVIRONMENT == "LOCAL" and ISOLATE_DOCUMENTS:
            print("       ⚠️ MERGE_EACH already wrote every quarter whose filing produced all "
                  "three")
            print("          statements. This is the SWEEP: those come back `identical to the "
                  "row")
            print("          already on disk`, and what §6 HELD is what this writes.")
        print()
        TALLY = pdf_ocr_batch.merge_batch(
            FOLDERS, apply=MERGE_APPLY, reports=MERGE_REPORTS,
            force_empty_band=FORCE_EMPTY_BAND)
        print()
        if not MERGE_APPLY:
            print("nothing was written. Set MERGE_APPLY = True to apply the plan above.")
            print("⚠️ AND THE PLAN ABOVE UNDERSTATES IT, BY CONSTRUCTION: with nothing written, a")
            print("   later period is planned against a span the earlier one has not "
                  "recorded yet,")
            print("   and reports the refusal it always would. A dry run cannot show a "
                  "second pass")
            print("   that depends on the first.")
        elif TALLY["written"]:
            print(f"{TALLY['written']} statement(s) reached raw_data/.../statements/ — §10 reads")
            print("the CSVs themselves, which is the only place the two can be told apart.")
        elif TALLY.get("already"):
            # ⚠️ **"0 WRITTEN" IS THE NORMAL OUTCOME OF THIS PASS ONCE §6 HAS ALREADY WRITTEN,
            # AND IT MUST NOT READ AS THE ALARM BELOW.** The branch after this one is `BND-1`'s
            # siren — a run that parsed and landed nothing — and printing it over a ticker
            # whose every quarter is on disk would train a reader to ignore the one message
            # that matters. What tells them apart is `already`: a statement re-planned against
            # disk and found unchanged is a CHECK that passed, not a refusal.
            print(f"0 written, {TALLY['already']} statement(s) already on disk unchanged — "
                  f"this pass")
            print("re-planned what §6 wrote and agreed with it. That is the sweep doing its "
                  "job.")
            if TALLY["skipped"]:
                print(f"⚠️ {TALLY['skipped']} statement(s) WERE refused — §8 says which and "
                      f"why. Those are the")
                print("   quarters §6 held back: a filing that produced two statements of "
                      "three.")
        else:
            # ⚠️ "THE RUN FINISHED" AND "THE CSV CHANGED" ARE DIFFERENT FACTS, and only the
            # second was ever the point. Read back from each folder's own `merge` block — the
            # structured record `record_merge` has just written — rather than from the lines
            # above, so this reports what a later reader gets and not what this cell printed.
            import collections                                # noqa: E402

            print("⚠️ NOTHING REACHED raw_data/.../statements/. Every accepted statement was")
            print("   refused, and these are the refusals, most common first:")
            WHY = collections.Counter(
                (_d["reason"] or "").split(" — ")[0].split(" because ")[0][:64]
                for _f in FOLDERS
                for _ev in (json.loads((Path(_f) / "metadata.json").read_text(encoding="utf-8"))
                            .get("merge") or {}).get("events", []) if _ev["applied"]
                for _d in _ev["decisions"] if _d["action"] != "write")
            for _reason, _n in (WHY.most_common(6)
                                or [("(no merge block — nothing was planned)", 0)]):
                print(f"     {_n:>4}  {_reason}")
            # ⚠️ THE ONE REFUSAL THAT CLOSES ON ITSELF — and since 2026-09-06 a quarter
            # with all three statements is past it before this branch can be reached, so
            # anything left here is a filing that produced TWO of three (or none).
            if any("band" in _r for _r in WHY):
                print("   ⚠️ `sane` band EMPTY is `BND-1`, and it is a LOOP: this ticker has no")
                print("      `pdf` row on disk, so `seed_history` builds no magnitude band, so")
                print("      every statement is refused, so there is still no CSV.")
                print("      ⚠️ A QUARTER WHOSE FILING PRODUCED ALL THREE STATEMENTS IS ALREADY "
                      "PAST THIS")
                print("         — both writers lift the band for it. What is refused here "
                      "produced two")
                print("         of three, which is a judgement about THAT filing and stays "
                      "yours: §8 says")
                print("         which statement is missing and why, and FORCE_EMPTY_BAND = True "
                      "writes the")
                print("         other two anyway. It LIFTS A REAL GUARD, so screen the figures "
                      "by arithmetic")
                print("         first (two statements agreeing on one figure, a printed subtotal")
                print("         closing) before quoting any of them.")
NB.end()

## 10 · Did it land? — the statement CSVs themselves

In [ ]:
# ── DID IT LAND? — the statement CSVs themselves ───────────────────────────
# ⚠️ Everything above reports what some process DECIDED; this reads what is on disk. The two
# came apart on a Kaggle round trip that finished green, wrote a complete run folder and created
# no CSV at all (`BND-1`, HOSE_BSR and again HOSE_CTG) — and no amount of log reading would have
# said so, because the merge that refused everything ran on the other machine.
NB.begin("landed", "the statement CSVs themselves — not what a process decided")
with NB.capture(nested=True):
    import csv                                            # noqa: E402

    from web_scraper import cafef_financials as fin       # noqa: E402

    # ⚠️ `CWD-1`, AND THIS CELL WALKED STRAIGHT INTO IT. `statement_path()` reads
    # `fin.STATEMENTS_DIR` at call time and its module default is RELATIVE — while the SETUP cell
    # `os.chdir`s to `src/kaggle_gpu`, where `kgpu` stages its payload. So the first version
    # of this cell reported `NO FILE` for a ticker whose three CSVs were on disk. The path
    # is PRINTED,
    # because a directory nobody names is a directory nobody checks.
    ROOT = job.use_data_root(job.DEFAULT_DATA_ROOT)

    TPL = (META or {}).get("inputs", {}).get("template") or TEMPLATE
    # ⚠️ KEYED BY (period, REPORT), not by period. A merge writes one statement of a quarter and
    # skips another — VCB Q1-2026 wrote its income statement and cash flow while its balance sheet
    # was `identical to the row already on disk` — so a period-only set credits this run with a row
    # it deliberately left alone.
    # ⚠️ **EVERY RUN FOLDER, NOT ONE — AND THIS COLUMN NEVER APPEARED ON A BATCH UNTIL
    # 2026-09-06.** It read `MERGE`, which only the one-process path ever assigns; a batch is
    # ONE FOLDER PER DOCUMENT and both writers record into their own through
    # `pdf_ocr_merge.record_merge` (§6's per-quarter upsert and §9's sweep alike), so there
    # was nothing to read and `<- N from this run` was silently never printed. A folder
    # without `metadata.json` is an interrupted child and is skipped, exactly as §9 skips it.
    import json                                          # noqa: E402

    DECIDED = [d
               for _f in (Path(_x) for _x in FOLDERS)
               if (_f / "metadata.json").is_file()
               for ev in (json.loads((_f / "metadata.json").read_text(encoding="utf-8"))
                          .get("merge") or {}).get("events", [])
               for d in ev["decisions"] if d["action"] == "write" and ev["applied"]]
    MINE = {(d["period"], d["report"]) for d in DECIDED}
    # ⚠️ **WHICH OF THIS RUN'S ROWS `sane` NEVER JUDGED — `band == 0` and nothing else.**
    # A quarter whose filing produced all three statements is written band or no band
    # (`BND-1` is a loop, not a guard — §1's MERGE_EACH), so this is the price of that
    # decision and it has to be READABLE rather than merely taken. ⚠️ `0` is "the band was
    # empty"; a MISSING `band` key is a run folder written before 2026-09-06, which is
    # "nobody recorded it" and NOT the same claim (§5 rule 2) — so the test is `== 0`, and an
    # older artefact reports nothing here rather than reporting everything.
    UNGUARDED = sorted({(d["period"], d["report"]) for d in DECIDED if d.get("band") == 0})
    if UNGUARDED:
        print(f"⚠️ {len(UNGUARDED)} of the {len(MINE)} statement(s) this run wrote PASSED NO "
              f"MAGNITUDE GUARD:")
        print("   `seed_history` had no `pdf` row on disk to rebuild a band from, so `sane` "
              "failed open.")
        for _p, _r in UNGUARDED[:12]:
            print(f"     {_p:10} {_r}")
        if len(UNGUARDED) > 12:
            print(f"     … and {len(UNGUARDED) - 12} more")
        print("   ⚠️ SCREEN THESE BY ARITHMETIC BEFORE QUOTING ANY OF THEM — two statements "
              "agreeing on")
        print("      one figure, a printed subtotal closing. This is what `sane` would have "
              "caught:")
        print("      an OCR misread by three orders of magnitude reads like a figure "
              "(`TSS-1`'s BSR")
        print("      Q3-2019 was 361,884,738 against another layer's 361,884,738,267).")
        print("   The same list is in each run folder's `merge` block, per decision, as "
              "`band: 0`.")
        print("")
    if TPL is None:
        print("no template resolved — run the cells above first")
    else:
        print(f"{ROOT / 'financials' / 'statements' / TPL}   {EXCHANGE}_{SYMBOL}")
        print()
        ANY = False
        for _report in fin.REPORTS:
            _path = Path(fin.statement_path(TPL, _report, EXCHANGE, SYMBOL))
            if not _path.is_file():
                print(f"  {_report:18} ⚠️ NO FILE — {_path.name} does not exist")
                continue
            ANY = True
            with open(_path, encoding="utf-8-sig") as _f:
                _rows = list(csv.DictReader(_f))
            _src = {}
            for _r in _rows:
                _src[_r.get("source", "")] = _src.get(_r.get("source", ""), 0) + 1
            _mine = [_r for _r in _rows if _r.get("source") == "pdf"
                     and (_r["period"], _report) in MINE]
            print(f"  {_report:18} {len(_rows):>3} quarters   "
                  + "  ".join(f"{k}={v}" for k, v in sorted(_src.items()))
                  + (f"   <- {len(_mine)} from this run" if _mine else ""))
            # ⚠️ Rule 24: a financial statement comes from the filing PDF and from nothing else.
            # A `cafef` row is an HTML transcription and must not be in this file.
            if _src.get("cafef"):
                print(f"       ⚠️ {_src['cafef']} row(s) read `source=cafef` — an HTML "
                      f"transcription. §5 rule 24 forbids it.")
        if not ANY:
            print()
            print("  ⚠️ THIS TICKER HAS NO STATEMENT CSV AT ALL. The parse is in the run folder "
                  "and")
            print("     nothing was upserted — the MERGE section above says which refusal "
                  "stopped it.")
NB.end()

## 11 · Repair one row — scoped, deliberate, read the diff first

In [ ]:
# ── REPAIR ONE ROW — scoped, deliberate, and read the diff first ──────────
# ⚠️ THE ONLY WAY THIS NOTEBOOK OVERWRITES A GOOD-LOOKING `pdf` ROW. `pdf_ocr_merge` refuses a
# figure that DIFFERS from a `pdf` row on disk, because two runs disagreeing is not settled by
# preferring the newer one. That refusal is lifted here for the NAMED pairs only — never for the
# run — and `merge_run`'s own `periods`/`reports` filter is what scopes it.
# ⚠️ A BACKUP is taken before any write. Diff EVERY COLUMN afterwards, not the figures: three
# separate runs in this repo lost only a `publish_date` and a figures-only diff called each of
# them clean (CLAUDE.md §6-2-quatervicies, §6-2-quinvicies, §6-2-quadragies).
NB.begin("repair", f"{len(REPAIR)} scoped pair(s)   apply={REPAIR_APPLY}")
with NB.capture(nested=True):
    if LATEST is not None and REPAIR:
        from web_scraper import pdf_ocr_merge                 # noqa: E402

        HOW = "APPLY" if REPAIR_APPLY else "PLAN"
        print(f"{HOW} — {len(REPAIR)} scoped repair(s) from {LATEST.name}")
        print()
        for _period, _report in REPAIR:
            print(f"── {_period} {_report} " + "─" * 46)
            _rep = pdf_ocr_merge.merge_run(
                LATEST, apply=REPAIR_APPLY, periods=[_period], reports=[_report],
                force_differs=True, force_empty_band=FORCE_EMPTY_BAND)
            if getattr(_rep, "backup", None):
                print(f"   backup: {_rep.backup}")
        if not REPAIR_APPLY:
            print()
            print("nothing was written. Set REPAIR_APPLY = True to apply the plan above.")
    elif LATEST is not None:
        print("REPAIR is empty — no row already on disk was replaced.")
        print("  A statement this run parsed that disk already holds as `pdf` was refused as")
        print("  DIFFERS and left alone. That is the default and usually right; name the")
        print("  (quarter, statement) pair in REPAIR only once the FILING has settled which")
        print("  reading is correct.")
NB.done("end of the notebook")